In [ ]:
# Elektrotechnik Spickzettel
# Alle Imports die du brauchst:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lti, step

<div style="text-align: center; margin-top: 100px;">

<h1 style="font-size: 48px; color: #003366;">Spickzettel QV 2026</h1>
<h2 style="font-size: 32px; color: #0055a5;">Elektroniker EFZ – Schweiz</h2>

<p style="font-size: 20px; margin-top: 50px;">
<strong>Erstellt am:</strong> 03.01.2026<br>
<strong>Geändert am:</strong> 04.01.2026<br> 
<strong>Geändert am:</strong> 03.03.2026<br> 
<strong>Version:</strong> v0.1.2<br>
<strong>Autor:</strong> Alvar Bolliger
</p>

<p style="font-size: 16px; margin-top: 30px; color: #555555;">
Hinweis: Dies ist ein persönlicher Spickzettel für die QV 2026. <br>
Interaktive Grafiken und Berechnungen sind im Notebook eingebettet.
</p>

</div>

## 📋 Inhaltsverzeichnis

1. [Grundgrössen & Ohm'sches Gesetz](#1-grundgrössen--ohmsches-gesetz)
   - [1.1 Elektrische Grundgrössen](#11-elektrische-grundgrössen)
   - [1.2 Serien- und Parallelschaltung](#12-serien--und-parallelschaltung)
   - [1.3 Spannungsteiler / Stromteiler](#13-spannungsteiler--stromteiler)
2. [Kapazitor (C)](#2-kapazitor-c)
   - [2.1 Kapazität & Grundgleichungen](#21-kapazität--grundgleichungen)
   - [2.2 Seriell/Parallel geschaltete Kondensatoren](#22-serielllparallel-geschaltete-kondensatoren)
3. [Spule (L)](#3-spule-l)
   - [3.1 Induktivität & Spannung](#31-induktivität--spannung)
   - [3.2 Seriell/Parallel geschaltete Spulen](#32-serielllparallel-geschaltete-spulen)
4. [RC-Schaltungen (Zeitbereich)](#4-rc-schaltungen-zeitbereich)
   - [4.1 RC-Lade-/Entladekurve](#41-rc-lade--entladekurve)
   - [4.2 RC-Integrations-/Differentiereigenschaft](#42-rc-integrations--differentiereigenschaft)
5. [RL-Schaltungen (Zeitbereich)](#5-rl-schaltungen-zeitbereich)
   - [5.1 Stromaufbau und Abbau in RL](#51-stromaufbau-und-abbau-in-rl)
6. [RLC-Schwingkreise](#6-rlc-schwingkreise)
   - [6.1 Ungedämpfter Schwingkreis](#61-ungedämpfter-schwingkreis-ideales-lc)
   - [6.2 Gedämpfter RLC-Reihenschwingkreis](#62-gedämpfter-rlc-reihenschwingkreis)
7. [Wechselstrom, Impedanzen & Zeiger](#7-wechselstrom-impedanzen--zeiger)
   - [7.1 Komplexe Impedanzen](#71-komplexe-impedanzen)
   - [7.2 Effektivwerte & Leistung im AC](#72-effektivwerte--leistung-im-ac)
8. [Filter (RC, RL, RLC)](#8-filter-rc-rl-rlc)
   - [8.1 RC-Tiefpass 1. Ordnung](#81-rc-tiefpass-1-ordnung)
   - [8.2 RC-Hochpass 1. Ordnung](#82-rc-hochpass-1-ordnung)
   - [8.3 RL-Tiefpass / Hochpass](#83-rl-tiefpass--hochpass)
   - [8.4 RLC-Bandpass / Bandsperre](#84-rlc-bandpass--bandsperre)
9. [Netzwerke – Knoten-/Maschensatz](#9-netzwerke--knoten--maschensatz)
   - [9.1 Kirchhoff'sche Gesetze](#91-kirchhoffsche-gesetze)
   - [9.2 Thevenin / Norton](#92-thevenin--norton)

In [ ]:
# ============================================================
# SETUP – Bitte zuerst ausführen!
# ============================================================
import re
import ipywidgets as widgets
from IPython.display import display, HTML
import math

# --- SI-Präfix Parser ---
SI_PREFIXES = {
    'p': 1e-12, 'n': 1e-9,
    'u': 1e-6,  'µ': 1e-6,
    'm': 1e-3,  '':  1.0,
    'k': 1e3,   'M': 1e6,  'G': 1e9
}

def p(val_str):
    """Parst Zahlen mit optionalem SI-Präfix: z.B. 10k, 4.7u, 100n"""
    if val_str is None: return None
    val_str = val_str.strip()
    if val_str == "": return None
    match = re.fullmatch(r'([-+]?\d*\.?\d+)\s*([pnumkMGµ]?)', val_str)
    if not match:
        raise ValueError(f"Ungültiger Wert: '{val_str}'")
    value, prefix = match.groups()
    return float(value) * SI_PREFIXES[prefix]

def fmt(val, unit=''):
    """Formatiert Zahlen schön mit SI-Präfix."""
    if val == 0:
        return f"0 {unit}"
    abs_val = abs(val)
    prefixes = [('G', 1e9), ('M', 1e6), ('k', 1e3), ('', 1),
                ('m', 1e-3), ('µ', 1e-6), ('n', 1e-9), ('p', 1e-12)]
    for name, factor in prefixes:
        if abs_val >= factor:
            return f"{val/factor:.4g} {name}{unit}"
    return f"{val:.4e} {unit}"

def make_input(description, placeholder='leer lassen = berechnen'):
    return widgets.Text(
        description=description,
        placeholder=placeholder,
        layout=widgets.Layout(width='320px'),
        style={'description_width': '130px'}
    )

def make_button(label='🔢 Berechnen'):
    return widgets.Button(
        description=label,
        button_style='primary',
        layout=widgets.Layout(width='180px', margin='8px 0')
    )

## 1. Widerstand (R)

---

### 1.1 Die Grundlagen

Der elektrische **Widerstand** $R$ gibt an, wie stark ein Bauteil den Stromfluss hemmt.

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Widerstand | $R$ | Ohm [Ω] |
| Spannung | $U$ | Volt [V] |
| Strom | $I$ | Ampere [A] |
| Leistung | $P$ | Watt [W] |

### Ohmsches Gesetz
$$U = R \cdot I$$

In [ ]:
# ============================================================
# RECHNER 1.1: Ohmsches Gesetz  U = R · I
# ============================================================
display(HTML("<h3>🔢 Rechner 1 – Ohmsches Gesetz</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

U_ohm = make_input('U [V]  =')
R_ohm = make_input('R [Ω]  =')
I_ohm = make_input('I [A]  =')
btn_ohm = make_button()
out_ohm = widgets.Output()

def calc_ohm(_=None):
    out_ohm.clear_output()
    with out_ohm:
        try:
            U = p(U_ohm.value)
            R = p(R_ohm.value)
            I = p(I_ohm.value)
            vals = {'U': U, 'R': R, 'I': I}
            none_fields = [k for k, v in vals.items() if v is None]
            if len(none_fields) == 0:
                print("❌ Ein Feld leer lassen!")
                return
            if len(none_fields) > 1:
                print("❌ Nur ein Feld leer lassen!")
                return
            miss = none_fields[0]
            if miss == 'U': res = R * I;   label = 'U'
            elif miss == 'R': res = U / I; label = 'R'
            else:             res = U / R; label = 'I'
            units = {'U': 'V', 'R': 'Ω', 'I': 'A'}
            print(f"{label} = {fmt(res, units[label])}")
            print(f"({res:.6e} {units[label]})")
        except Exception as e:
            print("❌ Fehler:", e)

btn_ohm.on_click(calc_ohm)
for field in [U_ohm, R_ohm, I_ohm]:
    field.observe(calc_ohm, names='value')

display(widgets.VBox([U_ohm, R_ohm, I_ohm, btn_ohm, out_ohm]))

### 1.2 Leistung

$$P = U \cdot I$$

**Energie:** $W = P \cdot t$ \[Joule = Watt · Sekunden\]

In [ ]:
# ============================================================
# RECHNER 1.2: Leistung  P = U · I
# ============================================================
display(HTML("<h3>🔢 Rechner 2 – Elektrische Leistung</h3>"
             "<i>Ein Feld leer lassen → wird berechnet. Für P=I²·R oder P=U²/R: P + R + {I oder U}</i>"))

P_inp = make_input('P [W]  =')
U_inp = make_input('U [V]  =')
I_inp = make_input('I [A]  =')
R_inp = make_input('R [Ω]  =')
btn_P = make_button()
out_P = widgets.Output()

def calc_P(_=None):
    out_P.clear_output()
    with out_P:
        try:
            PP = p(P_inp.value)
            UU = p(U_inp.value)
            II = p(I_inp.value)
            RR = p(R_inp.value)

            # P = U·I
            if PP is None and UU is not None and II is not None:
                print(f"P = U·I = {fmt(UU*II, 'W')}")
            elif UU is None and PP is not None and II is not None:
                print(f"U = P/I = {fmt(PP/II, 'V')}")
            elif II is None and PP is not None and UU is not None:
                print(f"I = P/U = {fmt(PP/UU, 'A')}")
            # P = I²·R
            elif PP is None and II is not None and RR is not None:
                print(f"P = I²·R = {fmt(II**2 * RR, 'W')}")
            elif RR is None and PP is not None and II is not None:
                print(f"R = P/I² = {fmt(PP/II**2, 'Ω')}")
            elif II is None and PP is not None and RR is not None:
                print(f"I = √(P/R) = {fmt(math.sqrt(PP/RR), 'A')}")
            # P = U²/R
            elif PP is None and UU is not None and RR is not None:
                print(f"P = U²/R = {fmt(UU**2 / RR, 'W')}")
            elif RR is None and PP is not None and UU is not None:
                print(f"R = U²/P = {fmt(UU**2 / PP, 'Ω')}")
            elif UU is None and PP is not None and RR is not None:
                print(f"U = √(P·R) = {fmt(math.sqrt(PP*RR), 'V')}")
            else:
                print("ℹ️ Gib genau 2 Werte ein (z.B. U + I, oder I + R, oder U + R)")
        except Exception as e:
            print("❌ Fehler:", e)

btn_P.on_click(calc_P)
for field in [P_inp, U_inp, I_inp, R_inp]:
    field.observe(calc_P, names='value')

display(widgets.VBox([P_inp, U_inp, I_inp, R_inp, btn_P, out_P]))

### 1.3 Serienschaltung

$$R_{ges} = R_1 + R_2 + R_3 + \ldots$$

**Eigenschaften:**
- Überall **gleicher Strom**: $I_{ges} = I_1 = I_2 = I_3$
- Spannungen addieren sich: $U_{ges} = U_1 + U_2 + U_3$
- Gesamtwiderstand **grösser** als jeder Einzelwiderstand

**Spannungsteiler:**
$$U_x = U_{ges} \cdot \frac{R_x}{R_{ges}}$$

In [ ]:
# ============================================================
# RECHNER 1.3: Reihenschaltung (bis 5 Widerstände)
# ============================================================
display(HTML("<h3>🔢 Rechner 3 – Reihenschaltung</h3>"
             "<i>Bis zu 5 Widerstände. Leere Felder werden ignoriert. Mindestens 2 Werte.</i>"))

r_fields = [make_input(f'R{i+1} [Ω] =') for i in range(5)]
U_reihe = make_input('U_ges [V] =')
btn_reihe = make_button()
out_reihe = widgets.Output()

def calc_reihe(_=None):
    out_reihe.clear_output()
    with out_reihe:
        try:
            Rs = [p(f.value) for f in r_fields]
            Rs_valid = [r for r in Rs if r is not None]
            U_ges = p(U_reihe.value)
            if len(Rs_valid) < 1:
                print("ℹ️ Mindestens 1 Widerstand eingeben")
                return
            R_ges = sum(Rs_valid)
            print(f"R_ges = {fmt(R_ges, 'Ω')}  ({R_ges:.6e} Ω)")
            if U_ges is not None and R_ges > 0:
                I = U_ges / R_ges
                print(f"I = {fmt(I, 'A')} (bei U_ges = {fmt(U_ges, 'V')})")
                for i, r in enumerate(Rs):
                    if r is not None:
                        U_teil = I * r
                        print(f"U_R{i+1} = {fmt(U_teil, 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_reihe.on_click(calc_reihe)
for f in r_fields + [U_reihe]:
    f.observe(calc_reihe, names='value')

display(widgets.VBox(r_fields + [U_reihe, btn_reihe, out_reihe]))

---
## 1.4. Parallelschaltung

$$\frac{1}{R_{ges}} = \frac{1}{R_1} + \frac{1}{R_2} + \frac{1}{R_3} + \ldots$$

**Für 2 Widerstände (Produktformel):**
$$R_{ges} = \frac{R_1 \cdot R_2}{R_1 + R_2}$$

**Eigenschaften:**
- Überall **gleiche Spannung**: $U_{ges} = U_1 = U_2 = U_3$
- Ströme addieren sich: $I_{ges} = I_1 + I_2 + I_3$
- Gesamtwiderstand **kleiner** als der kleinste Einzelwiderstand

**Stromteiler (2 Widerstände):**
$$I_1 = I_{ges} \cdot \frac{R_2}{R_1 + R_2}$$

In [ ]:
# ============================================================
# RECHNER 1.4: Parallelschaltung (bis 5 Widerstände)
# ============================================================
display(HTML("<h3>🔢 Rechner 4 – Parallelschaltung</h3>"
             "<i>Bis zu 5 Widerstände. Leere Felder werden ignoriert.</i>"))

rp_fields = [make_input(f'R{i+1} [Ω] =') for i in range(5)]
U_para = make_input('U_ges [V] =')
btn_para = make_button()
out_para = widgets.Output()

def calc_para(_=None):
    out_para.clear_output()
    with out_para:
        try:
            Rs = [p(f.value) for f in rp_fields]
            Rs_valid = [r for r in Rs if r is not None]
            U_ges = p(U_para.value)
            if len(Rs_valid) < 1:
                print("ℹ️ Mindestens 1 Widerstand eingeben")
                return
            R_ges = 1 / sum(1/r for r in Rs_valid)
            print(f"R_ges = {fmt(R_ges, 'Ω')}  ({R_ges:.6e} Ω)")
            if U_ges is not None:
                I_ges = U_ges / R_ges
                print(f"   I_ges = {fmt(I_ges, 'A')}  (bei U_ges = {fmt(U_ges, 'V')})")
                for i, r in enumerate(Rs):
                    if r is not None:
                        I_teil = U_ges / r
                        print(f"   I_R{i+1}  = {fmt(I_teil, 'A')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_para.on_click(calc_para)
for f in rp_fields + [U_para]:
    f.observe(calc_para, names='value')

display(widgets.VBox(rp_fields + [U_para, btn_para, out_para]))

---
## 1.5. Temperaturabhängigkeit

$$R_T = R_{20} \cdot [1 + \alpha_{20} \cdot (T - 20°C)]$$

- **$\alpha_{20}$** = Temperaturkoeffizient (bei 20°C Referenz)
- Metalle: $\alpha > 0$ → Widerstand steigt mit Temperatur (PTC)
- NTC-Widerstände: $\alpha < 0$ → Widerstand sinkt mit Temperatur

In [ ]:
# ============================================================
# RECHNER 1.5: Temperaturabhängigkeit  R_T = R_20 · [1 + α·(T-20)]
# ============================================================
display(HTML("<h3>🔢 Rechner 5 – Temperaturabhängigkeit</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

R20_inp  = make_input('R₂₀ [Ω]  =')
alpha_inp = make_input('α₂₀ [1/K] =', 'z.B. 0.00393 für Cu')
T_inp    = make_input('T [°C]   =')
RT_inp   = make_input('R_T [Ω]  =')

# Materialvorauswahl
mat_select = widgets.Dropdown(
    options=[('-- Material wählen --', None),
             ('Kupfer Cu', 0.00393), ('Aluminium Al', 0.00403),
             ('Eisen Fe', 0.00650), ('Konstantan', 0.00001), ('Silber Ag', 0.00380)],
    description='Material:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

def set_alpha(change):
    if change['new'] is not None:
        alpha_inp.value = str(change['new'])
mat_select.observe(set_alpha, names='value')

btn_temp = make_button()
out_temp = widgets.Output()

def calc_temp(_=None):
    out_temp.clear_output()
    with out_temp:
        try:
            R20 = p(R20_inp.value)
            alpha = p(alpha_inp.value)
            T = p(T_inp.value)
            RT = p(RT_inp.value)
            vals = {'R₂₀': R20, 'α₂₀': alpha, 'T': T, 'R_T': RT}
            none_fields = [k for k, v in vals.items() if v is None]
            if len(none_fields) == 0:
                print("❌ Ein Feld leer lassen!")
                return
            if len(none_fields) > 1:
                print("❌ Nur ein Feld leer lassen!")
                return
            miss = none_fields[0]
            if miss == 'R_T':
                res = R20 * (1 + alpha * (T - 20))
                print(f"R_T  = {fmt(res, 'Ω')}")
            elif miss == 'R₂₀':
                res = RT / (1 + alpha * (T - 20))
                print(f"R₂₀  = {fmt(res, 'Ω')}")
            elif miss == 'T':
                res = ((RT / R20) - 1) / alpha + 20
                print(f"T    = {res:.4g} °C")
            elif miss == 'α₂₀':
                res = ((RT / R20) - 1) / (T - 20)
                print(f"α₂₀  = {res:.6g} 1/K")
        except Exception as e:
            print("❌ Fehler:", e)

btn_temp.on_click(calc_temp)
for f in [R20_inp, alpha_inp, T_inp, RT_inp]:
    f.observe(calc_temp, names='value')

display(widgets.VBox([mat_select, R20_inp, alpha_inp, T_inp, RT_inp, btn_temp, out_temp]))

---
## 1.6. Spezifischer Widerstand

$$R = \rho \cdot \frac{l}{A}$$

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Widerstand | $R$ | Ω |
| Spez. Widerstand | $\rho$ (rho) | Ω·mm²/m |
| Länge | $l$ | m |
| Querschnitt | $A$ | mm² |

In [ ]:
# ============================================================
# RECHNER 1.6: Leitungswiderstand  R = ρ · l / A
# ============================================================
display(HTML("<h3>🔢 Rechner 6 – Leitungswiderstand</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

rho_select = widgets.Dropdown(
    options=[('-- Material wählen --', None),
             ('Kupfer Cu', 0.0178), ('Aluminium Al', 0.0283),
             ('Eisen Fe', 0.100),   ('Konstantan', 0.500)],
    description='Material:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

R_leitung  = make_input('R [Ω]        =')
rho_inp    = make_input('ρ [Ω·mm²/m]  =', 'z.B. 0.0178 für Cu')
l_inp      = make_input('l [m]         =')
A_inp      = make_input('A [mm²]       =')

def set_rho(change):
    if change['new'] is not None:
        rho_inp.value = str(change['new'])
rho_select.observe(set_rho, names='value')

btn_leit = make_button()
out_leit = widgets.Output()

def calc_leit(_=None):
    out_leit.clear_output()
    with out_leit:
        try:
            R  = p(R_leitung.value)
            rho = p(rho_inp.value)
            l  = p(l_inp.value)
            A  = p(A_inp.value)
            vals = {'R': R, 'ρ': rho, 'l': l, 'A': A}
            none_fields = [k for k, v in vals.items() if v is None]
            if len(none_fields) == 0:
                print("❌ Ein Feld leer lassen!")
                return
            if len(none_fields) > 1:
                print("❌ Nur ein Feld leer lassen!")
                return
            miss = none_fields[0]
            # R = rho * l / A  (rho in Ω·mm²/m, l in m, A in mm²)
            if miss == 'R':
                res = rho * l / A
                print(f"R = ρ·l/A = {fmt(res, 'Ω')}")
            elif miss == 'ρ':
                res = R * A / l
                print(f"ρ = R·A/l = {res:.6g} Ω·mm²/m")
            elif miss == 'l':
                res = R * A / rho
                print(f"l = R·A/ρ = {fmt(res, 'm')}")
            elif miss == 'A':
                res = rho * l / R
                print(f"A = ρ·l/R = {res:.6g} mm²")
        except Exception as e:
            print("❌ Fehler:", e)

btn_leit.on_click(calc_leit)
for f in [R_leitung, rho_inp, l_inp, A_inp]:
    f.observe(calc_leit, names='value')

display(widgets.VBox([rho_select, R_leitung, rho_inp, l_inp, A_inp, btn_leit, out_leit]))

---
## 1.7 Farbcode-Lesung (4-, 5- und 6-Ring Widerstände)

**4-Ring** (Standard):
```
Ring 1+2: Ziffern | Ring 3: Multiplikator | Ring 4: Toleranz
```

**5-Ring** (Präzisionswiderstand):
```
Ring 1+2+3: Ziffern | Ring 4: Multiplikator | Ring 5: Toleranz
```

**6-Ring** (Präzision + Temperaturkoeffizient):
```
Ring 1+2+3: Ziffern | Ring 4: Multiplikator | Ring 5: Toleranz | Ring 6: TK [ppm/K]
```

| Farbe | Ziffer | Multiplikator | Toleranz | TK [ppm/K] |
|-------|--------|---------------|----------|------------|
| Schwarz | 0 | ×1 | – | 250 |
| Braun | 1 | ×10 | ±1% | 100 |
| Rot | 2 | ×100 | ±2% | 50 |
| Orange | 3 | ×1k | – | 15 |
| Gelb | 4 | ×10k | – | 25 |
| Grün | 5 | ×100k | ±0.5% | – |
| Blau | 6 | ×1M | ±0.25% | 10 |
| Violett | 7 | ×10M | ±0.1% | 5 |
| Grau | 8 | ×100M | ±0.05% | – |
| Weiss | 9 | – | – | – |
| Gold | – | ×0.1 | ±5% | – |
| Silber | – | ×0.01 | ±10% | – |

In [ ]:
# ============================================================
# RECHNER 1.7: Widerstand-Farbcode Decoder (4 / 5 / 6 Ringe)
# ============================================================
display(HTML("<h3>🔢 Rechner 1.7 – Farbcode Decoder</h3>"))

COLORS = {
    'Schwarz': {'digit': 0, 'mult': 1,      'tol': None,     'tk': 250},
    'Braun':   {'digit': 1, 'mult': 10,     'tol': '±1%',    'tk': 100},
    'Rot':     {'digit': 2, 'mult': 100,    'tol': '±2%',    'tk': 50},
    'Orange':  {'digit': 3, 'mult': 1e3,    'tol': None,     'tk': 15},
    'Gelb':    {'digit': 4, 'mult': 1e4,    'tol': None,     'tk': 25},
    'Grün':    {'digit': 5, 'mult': 1e5,    'tol': '±0.5%',  'tk': None},
    'Blau':    {'digit': 6, 'mult': 1e6,    'tol': '±0.25%', 'tk': 10},
    'Violett': {'digit': 7, 'mult': 1e7,    'tol': '±0.1%',  'tk': 5},
    'Grau':    {'digit': 8, 'mult': 1e8,    'tol': '±0.05%', 'tk': None},
    'Weiss':   {'digit': 9, 'mult': None,   'tol': None,     'tk': None},
    'Gold':    {'digit': None, 'mult': 0.1, 'tol': '±5%',    'tk': None},
    'Silber':  {'digit': None, 'mult': 0.01,'tol': '±10%',   'tk': None},
}

digit_colors = [c for c, v in COLORS.items() if v['digit'] is not None]
mult_colors  = [c for c, v in COLORS.items() if v['mult']  is not None]
tol_colors   = [c for c, v in COLORS.items() if v['tol']   is not None]
tk_colors    = [c for c, v in COLORS.items() if v['tk']    is not None]

dd_style  = {'description_width': '160px'}
dd_layout = widgets.Layout(width='340px')

# Ringanzahl wählen
ring_count = widgets.ToggleButtons(
    options=['4 Ringe', '5 Ringe', '6 Ringe'],
    description='Ausführung:',
    style={'description_width': '160px', 'button_width': '90px'}
)

# Alle Ringe vorbereiten
r1 = widgets.Dropdown(options=digit_colors, description='Ring 1 (Ziffer 1):', style=dd_style, layout=dd_layout)
r2 = widgets.Dropdown(options=digit_colors, description='Ring 2 (Ziffer 2):', style=dd_style, layout=dd_layout)
r3 = widgets.Dropdown(options=digit_colors, description='Ring 3 (Ziffer 3):', style=dd_style, layout=dd_layout)
r4 = widgets.Dropdown(options=mult_colors,  description='Ring 4 (Multiplik.):', style=dd_style, layout=dd_layout)
r5 = widgets.Dropdown(options=tol_colors,   description='Ring 5 (Toleranz):', style=dd_style, layout=dd_layout, value='Gold')
r6 = widgets.Dropdown(options=tk_colors,    description='Ring 6 (TK ppm/K):', style=dd_style, layout=dd_layout)

# Ring-Boxen für 4 / 5 / 6
box_4 = widgets.VBox([r1, r2,
                      widgets.Dropdown(options=mult_colors, description='Ring 3 (Multiplik.):', style=dd_style, layout=dd_layout),
                      widgets.Dropdown(options=tol_colors,  description='Ring 4 (Toleranz):',  style=dd_style, layout=dd_layout, value='Gold')])
box_5 = widgets.VBox([r1, r2, r3, r4, r5])
box_6 = widgets.VBox([r1, r2, r3, r4, r5, r6])

# eigene refs für 4-Ring (separate Dropdowns im box_4)
r3_4 = box_4.children[2]
r4_4 = box_4.children[3]

out_farb = widgets.Output()
container = widgets.VBox([])

def update_view(_=None):
    if ring_count.value == '4 Ringe':
        container.children = [box_4, out_farb]
    elif ring_count.value == '5 Ringe':
        container.children = [box_5, out_farb]
    else:
        container.children = [box_6, out_farb]
    calc_farb()

def calc_farb(_=None):
    out_farb.clear_output()
    with out_farb:
        try:
            mode = ring_count.value
            if mode == '4 Ringe':
                d1   = COLORS[r1.value]['digit']
                d2   = COLORS[r2.value]['digit']
                mult = COLORS[r3_4.value]['mult']
                tol  = COLORS[r4_4.value]['tol']
                R    = (d1 * 10 + d2) * mult
                tk_str = ''
            elif mode == '5 Ringe':
                d1   = COLORS[r1.value]['digit']
                d2   = COLORS[r2.value]['digit']
                d3   = COLORS[r3.value]['digit']
                mult = COLORS[r4.value]['mult']
                tol  = COLORS[r5.value]['tol']
                R    = (d1 * 100 + d2 * 10 + d3) * mult
                tk_str = ''
            else:  # 6 Ringe
                d1   = COLORS[r1.value]['digit']
                d2   = COLORS[r2.value]['digit']
                d3   = COLORS[r3.value]['digit']
                mult = COLORS[r4.value]['mult']
                tol  = COLORS[r5.value]['tol']
                tk   = COLORS[r6.value]['tk']
                R    = (d1 * 100 + d2 * 10 + d3) * mult
                tk_str = f"   TK    = {tk} ppm/K" if tk else ''

            tol_num = float(tol.strip('±%'))
            R_min = R * (1 - tol_num / 100)
            R_max = R * (1 + tol_num / 100)
            print(f"R      = {fmt(R, 'Ω')}  {tol}")
            print(f"Bereich: {fmt(R_min, 'Ω')} … {fmt(R_max, 'Ω')}")
            if tk_str:
                print(tk_str)
        except Exception as e:
            print("❌ Fehler:", e)

ring_count.observe(update_view, names='value')
for w in [r1, r2, r3, r4, r5, r6, r3_4, r4_4]:
    w.observe(calc_farb, names='value')

update_view()
display(widgets.VBox([ring_count, container]))

---
## 1.8. Spannungsteiler (unbelastet)

$$U_2 = U_{ges} \cdot \frac{R_2}{R_1 + R_2}$$

Oder allgemein:
$$U_x = U_{ges} \cdot \frac{R_x}{R_{ges}}$$

> ⚠️ **Wichtig:** Diese Formel gilt nur für den **unbelasteten** Spannungsteiler (kein Strom wird abgenommen)!

In [ ]:
# ============================================================
# RECHNER 1.8: Spannungsteiler (unbelastet)
# ============================================================
display(HTML("<h3>🔢 Rechner 8 – Spannungsteiler (unbelastet)</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

Uges_st = make_input('U_ges [V] =')
R1_st   = make_input('R1 [Ω]   =')
R2_st   = make_input('R2 [Ω]   =')
U2_st   = make_input('U2 [V]   =')
btn_st  = make_button()
out_st  = widgets.Output()

def calc_st(_=None):
    out_st.clear_output()
    with out_st:
        try:
            Ug = p(Uges_st.value)
            R1 = p(R1_st.value)
            R2 = p(R2_st.value)
            U2 = p(U2_st.value)
            vals = {'U_ges': Ug, 'R1': R1, 'R2': R2, 'U2': U2}
            none_fields = [k for k, v in vals.items() if v is None]
            if len(none_fields) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(none_fields) > 1 else "❌ Ein Feld leer lassen!")
                return
            miss = none_fields[0]
            if miss == 'U2':
                res = Ug * R2 / (R1 + R2)
                print(f"U2    = {fmt(res, 'V')}")
                print(f"U1    = {fmt(Ug - res, 'V')}")
                I = Ug / (R1 + R2)
                print(f"   I     = {fmt(I, 'A')}")
            elif miss == 'U_ges':
                res = U2 * (R1 + R2) / R2
                print(f"U_ges = {fmt(res, 'V')}")
            elif miss == 'R1':
                res = R2 * (Ug - U2) / U2
                print(f"R1    = {fmt(res, 'Ω')}")
            elif miss == 'R2':
                res = R1 * U2 / (Ug - U2)
                print(f"R2    = {fmt(res, 'Ω')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_st.on_click(calc_st)
for f in [Uges_st, R1_st, R2_st, U2_st]:
    f.observe(calc_st, names='value')

display(widgets.VBox([Uges_st, R1_st, R2_st, U2_st, btn_st, out_st]))

---
## 1.9 Leitwert und Siemens

Der **Leitwert** $G$ ist der Kehrwert des Widerstands.
Er gibt an wie gut ein Bauteil den Strom leitet.

$$G = \frac{1}{R} \qquad [S] = \left[\frac{1}{\Omega}\right]$$

$$I = G \cdot U \qquad G_{ges,parallel} = G_1 + G_2 + G_3 + \ldots$$

> Parallelschaltung mit Leitwerten ist einfacher als mit Widerständen –
> Leitwerte addieren sich direkt!

In [ ]:
# ============================================================
# RECHNER 1.9: Leitwert G = 1/R
# ============================================================
display(HTML("<h3>🔢 Rechner 1.9 – Leitwert / Siemens</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

lw_R   = make_input('R [Ω]  =')
lw_G   = make_input('G [S]  =')
lw_U   = make_input('U [V]  =', 'optional → I berechnen')
btn_lw = make_button()
out_lw = widgets.Output()

def calc_lw(_=None):
    out_lw.clear_output()
    with out_lw:
        try:
            R = p(lw_R.value); G = p(lw_G.value); U = p(lw_U.value)
            vals = {'R': R, 'G': G}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld (R oder G) leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if m == 'G':
                res = 1 / R
                print(f"  G = 1/R = {fmt(res, 'S')}")
            else:
                res = 1 / G
                print(f"  R = 1/G = {fmt(res, 'Ω')}")
            Gv = G if G else res if m == 'G' else None
            if U and Gv:
                print(f"  I = G·U = {fmt(Gv * U, 'A')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_lw.on_click(calc_lw)
for fw in [lw_R, lw_G, lw_U]:
    fw.observe(calc_lw, names='value')
display(widgets.VBox([lw_R, lw_G, lw_U, btn_lw, out_lw]))

---
## 1.10 Stromdichte

Die **Stromdichte** $J$ beschreibt wie viel Strom pro Querschnittsfläche fliesst.
Zu hohe Stromdichte → Überhitzung der Leitung!

$$J = \frac{I}{A} \qquad \left[\frac{A}{mm^2}\right]$$

| Leitungstyp | Zul. Stromdichte |
|-------------|-----------------|
| Festverlegung (NYM) | bis $6\,A/mm^2$ |
| Flexible Leitung | bis $4\,A/mm^2$ |
| Kurzzeit / Anlauf | bis $10\,A/mm^2$ |
| Leiterplatte (PCB) | bis $20\,A/mm^2$ |

### Querschnitt wählen
$$A_{min} = \frac{I}{J_{max}}$$

In [ ]:
# ============================================================
# RECHNER 1.10: Stromdichte J = I/A
# ============================================================
display(HTML("<h3>🔢 Rechner 1.10 – Stromdichte</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

jmax_select = widgets.Dropdown(
    options=[('-- Leitungstyp wählen --', None),
             ('Festverlegung NYM', 6), ('Flexible Leitung', 4),
             ('Kurzzeit / Anlauf', 10), ('Leiterplatte PCB', 20)],
    description='Leitungstyp:', style={'description_width': '140px'},
    layout=widgets.Layout(width='340px')
)

sd_I   = make_input('I [A]      =')
sd_A   = make_input('A [mm²]    =', 'Querschnitt')
sd_J   = make_input('J [A/mm²]  =', 'leer = berechnen')
btn_sd = make_button()
out_sd = widgets.Output()

def set_jmax(change):
    if change['new'] is not None:
        sd_J.value = str(change['new'])
jmax_select.observe(set_jmax, names='value')

def calc_sd(_=None):
    out_sd.clear_output()
    with out_sd:
        try:
            I = p(sd_I.value); A = p(sd_A.value); J = p(sd_J.value)
            vals = {'I': I, 'A': A, 'J': J}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'J': res = I / A;  print(f"  J = I/A = {res:.4g} A/mm²")
            elif m == 'A': res = I / J;  print(f"  A = I/J = {res:.4g} mm²")
            else:          res = J * A;  print(f"  I = J·A = {fmt(res, 'A')}")
            Jv = J if J else res if m == 'J' else None
            if Jv:
                if   Jv <= 4:  print(f"  → Gut für flexible Leitungen")
                elif Jv <= 6:  print(f"  → Gut für Festverlegung")
                elif Jv <= 10: print(f"  ⚠️ Nur für Kurzzeit/Anlauf geeignet")
                else:          print(f"  ⚠️ Zu hohe Stromdichte – Querschnitt vergrössern!")
        except Exception as e:
            print("❌ Fehler:", e)

btn_sd.on_click(calc_sd)
for fw in [sd_I, sd_A, sd_J]:
    fw.observe(calc_sd, names='value')
display(widgets.VBox([jmax_select, sd_I, sd_A, sd_J, btn_sd, out_sd]))

---
# Kapitel 2: Kondensatoren

---
## 2.1 📖 Grundlagen des Kondensators

Ein **Kondensator** speichert elektrische Energie im elektrischen Feld zwischen zwei leitenden Platten.

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Kapazität | $C$ | Farad [F] |
| Ladung | $Q$ | Coulomb [C] |
| Spannung | $U$ | Volt [V] |

### Grundformel
$$Q = C \cdot U$$

> **Energie im Kondensator:**
> $$W = \frac{1}{2} \cdot C \cdot U^2$$

In [ ]:
# ============================================================
# RECHNER 2.1: Grundgrössen Q = C · U
# ============================================================
display(HTML("<h3>🔢 Rechner 2.1 – Grundgrössen Q / C / U</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

Q_inp = make_input('Q [C]  =', 'z.B. 100u, 4.7m')
C_inp = make_input('C [F]  =', 'z.B. 100n, 4.7u')
U_inp = make_input('U [V]  =')
btn_qcu = make_button()
out_qcu = widgets.Output()

def calc_qcu(_=None):
    out_qcu.clear_output()
    with out_qcu:
        try:
            Q = p(Q_inp.value); C = p(C_inp.value); U = p(U_inp.value)
            vals = {'Q': Q, 'C': C, 'U': U}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'Q': res = C * U;  print(f"Q = C·U  = {fmt(res, 'C')}")
            elif m == 'C': res = Q / U;  print(f"C = Q/U  = {fmt(res, 'F')}")
            else:          res = Q / C;  print(f"U = Q/C  = {fmt(res, 'V')}")
            # Energie immer mitausgeben wenn C und U bekannt
            Cv = C if C else res if m == 'C' else None
            Uv = U if U else res if m == 'U' else None
            if Cv and Uv:
                W = 0.5 * Cv * Uv**2
                print(f"   W = ½·C·U² = {fmt(W, 'J')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_qcu.on_click(calc_qcu)
for f in [Q_inp, C_inp, U_inp]:
    f.observe(calc_qcu, names='value')
display(widgets.VBox([Q_inp, C_inp, U_inp, btn_qcu, out_qcu]))


---
## 2.2 Plattenkondensator

$$C = \varepsilon_0 \cdot \varepsilon_r \cdot \frac{A}{d}$$

| Grösse | Symbol | Wert / Einheit |
|--------|--------|----------------|
| Permittivität Vakuum | $\varepsilon_0$ | $8.854 \times 10^{-12}$ F/m |
| Relative Permittivität | $\varepsilon_r$ | dimensionslos |
| Plattenfläche | $A$ | m² |
| Plattenabstand | $d$ | m |

| Material | $\varepsilon_r$ |
|----------|-----------------|
| Luft / Vakuum | 1.0 |
| Papier | 2.0–2.5 |
| Glas | 4–10 |
| Keramik | 10–10'000 |
| Polyester | 3.2 |


In [ ]:
# ============================================================
# RECHNER 2.2: Plattenkondensator  C = ε0·εr·A/d
# ============================================================
display(HTML("<h3>🔢 Rechner 2.2 – Plattenkondensator</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

EPS0 = 8.854e-12

eps_mat = widgets.Dropdown(
    options=[('-- Material wählen --', None), ('Luft/Vakuum', 1.0),
             ('Papier', 2.2), ('Polyester', 3.2), ('Glas', 6.0), ('Keramik', 100.0)],
    description='Material:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

C_plat  = make_input('C [F]   =', 'leer = berechnen')
A_plat  = make_input('A [m²]  =', 'z.B. 0.01')
d_plat  = make_input('d [m]   =', 'z.B. 1m, 0.5m')
er_plat = make_input('ε_r     =', 'z.B. 1 für Luft')
btn_plat = make_button()
out_plat = widgets.Output()

def set_er(change):
    if change['new'] is not None:
        er_plat.value = str(change['new'])
eps_mat.observe(set_er, names='value')

def calc_plat(_=None):
    out_plat.clear_output()
    with out_plat:
        try:
            C  = p(C_plat.value)
            A  = p(A_plat.value)
            d  = p(d_plat.value)
            er = p(er_plat.value)
            vals = {'C': C, 'A': A, 'd': d, 'ε_r': er}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'C':   res = EPS0 * er * A / d;    print(f"C   = {fmt(res, 'F')}")
            elif m == 'A':   res = C * d / (EPS0 * er);  print(f"A   = {fmt(res, 'm²')}")
            elif m == 'd':   res = EPS0 * er * A / C;    print(f"d   = {fmt(res, 'm')}")
            elif m == 'ε_r': res = C * d / (EPS0 * A);   print(f"ε_r = {res:.4g}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_plat.on_click(calc_plat)
for f in [C_plat, A_plat, d_plat, er_plat]:
    f.observe(calc_plat, names='value')
display(widgets.VBox([eps_mat, C_plat, A_plat, d_plat, er_plat, btn_plat, out_plat]))

---
## 2.3 Kondensatoren in Reihe und Parallel

### Parallelschaltung
$$C_{ges} = C_1 + C_2 + C_3 + \ldots$$
- Gleiche **Spannung** an allen Kondensatoren
- Kapazitäten addieren sich direkt

### Reihenschaltung
$$\frac{1}{C_{ges}} = \frac{1}{C_1} + \frac{1}{C_2} + \frac{1}{C_3} + \ldots$$
- Gleiche **Ladung** auf allen Kondensatoren
- Gesamtkapazität **kleiner** als kleinster Einzelwert

In [ ]:
# ============================================================
# RECHNER 2.3: Kondensatoren Reihe / Parallel
# ============================================================
display(HTML("<h3>🔢 Rechner 2.3 – Reihe & Parallel</h3>"
             "<i>Bis 5 Kondensatoren. Leere Felder werden ignoriert.</i>"))

schalt_toggle = widgets.ToggleButtons(
    options=['Parallelschaltung', 'Reihenschaltung'],
    description='Schaltung:', style={'description_width': '130px', 'button_width': '160px'}
)
c_fields = [make_input(f'C{i+1} [F] =', 'z.B. 100n, 4.7u') for i in range(5)]
U_kond   = make_input('U_ges [V] =', 'optional')
btn_kond = make_button()
out_kond = widgets.Output()

def calc_kond(_=None):
    out_kond.clear_output()
    with out_kond:
        try:
            Cs = [p(f.value) for f in c_fields]
            Cs_v = [c for c in Cs if c is not None]
            Uv   = p(U_kond.value)
            if len(Cs_v) < 1:
                print("ℹ️ Mindestens 1 Kondensator eingeben"); return

            if schalt_toggle.value == 'Parallelschaltung':
                C_ges = sum(Cs_v)
                print(f"C_ges (parallel) = {fmt(C_ges, 'F')}")
                if Uv:
                    print(f"   Q_ges = {fmt(C_ges * Uv, 'C')}")
                    for i, c in enumerate(Cs):
                        if c: print(f"   Q_C{i+1} = {fmt(c * Uv, 'C')}  (U = {fmt(Uv, 'V')})")
            else:
                C_ges = 1 / sum(1/c for c in Cs_v)
                print(f"C_ges (Reihe) = {fmt(C_ges, 'F')}")
                if Uv:
                    Q = C_ges * Uv
                    print(f"   Q     = {fmt(Q, 'C')}  (gleich auf allen)")
                    for i, c in enumerate(Cs):
                        if c: print(f"   U_C{i+1} = {fmt(Q / c, 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_kond.on_click(calc_kond)
schalt_toggle.observe(calc_kond, names='value')
for f in c_fields + [U_kond]:
    f.observe(calc_kond, names='value')
display(widgets.VBox([schalt_toggle] + c_fields + [U_kond, btn_kond, out_kond]))

---
## 2.4 RC-Glied – Zeitkonstante und Lade-/Entladekurve

$$\tau = R \cdot C$$

| Zeit | Ladespannung $U_C$ | Restspannung (Entladung) |
|------|--------------------|--------------------------|
| $1\tau$ | 63.2% von $U_0$ | 36.8% |
| $2\tau$ | 86.5% | 13.5% |
| $3\tau$ | 95.0% | 5.0% |
| $4\tau$ | 98.2% | 1.8% |
| $5\tau$ | 99.3% | 0.7% ≈ vollständig |

**Ladespannung:**
$$U_C(t) = U_0 \cdot \left(1 - e^{-t/\tau}\right)$$

**Entladespannung:**
$$U_C(t) = U_0 \cdot e^{-t/\tau}$$

**Ladestrom:**
$$i(t) = \frac{U_0}{R} \cdot e^{-t/\tau}$$

In [ ]:
# ============================================================
# RECHNER 2.4a: Zeitkonstante τ = R·C
# ============================================================
import math
display(HTML("<h3>🔢 Rechner 2.4a – Zeitkonstante τ = R·C</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

tau_R = make_input('R [Ω]  =')
tau_C = make_input('C [F]  =', 'z.B. 100n, 4.7u')
tau_T = make_input('τ [s]  =', 'z.B. 10m = 10ms')
btn_tau = make_button()
out_tau = widgets.Output()

def calc_tau(_=None):
    out_tau.clear_output()
    with out_tau:
        try:
            R = p(tau_R.value); C = p(tau_C.value); T = p(tau_T.value)
            vals = {'R': R, 'C': C, 'τ': T}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'τ': res = R * C;   print(f"τ = R·C = {fmt(res, 's')}")
            elif m == 'R': res = T / C;   print(f"R = τ/C = {fmt(res, 'Ω')}")
            else:          res = T / R;   print(f"C = τ/R = {fmt(res, 'F')}")
            tau_val = T if T else res
            print(f"\n   Ladetabelle:")
            for n in [1, 2, 3, 4, 5]:
                uc_p = (1 - math.exp(-n)) * 100
                ue_p = math.exp(-n) * 100
                print(f"   {n}τ = {fmt(n * tau_val, 's'):12s}  → Laden: {uc_p:.1f}%  Entladen: {ue_p:.1f}%")
        except Exception as e:
            print("❌ Fehler:", e)

btn_tau.on_click(calc_tau)
for f in [tau_R, tau_C, tau_T]:
    f.observe(calc_tau, names='value')
display(widgets.VBox([tau_R, tau_C, tau_T, btn_tau, out_tau]))

In [ ]:
# ============================================================
# RECHNER 2.4b: Spannung / Strom zu bestimmtem Zeitpunkt
# ============================================================
display(HTML("<h3>🔢 Rechner 2.4b – U_C und I zu Zeitpunkt t</h3>"
             "<i>R, C und t eingeben</i>"))

rc2_mode = widgets.ToggleButtons(
    options=['Laden', 'Entladen'],
    description='Vorgang:', style={'description_width': '130px', 'button_width': '100px'}
)
rc2_R  = make_input('R [Ω]  =')
rc2_C  = make_input('C [F]  =', 'z.B. 100n')
rc2_U0 = make_input('U₀ [V] =', 'Quellspannung')
rc2_t  = make_input('t [s]  =', 'z.B. 5m = 5ms')
btn_rc2 = make_button()
out_rc2 = widgets.Output()

def calc_rc2(_=None):
    out_rc2.clear_output()
    with out_rc2:
        try:
            R  = p(rc2_R.value);  C  = p(rc2_C.value)
            U0 = p(rc2_U0.value); t  = p(rc2_t.value)
            if any(x is None for x in [R, C, U0, t]):
                print("ℹ️ Alle 4 Felder ausfüllen"); return
            tau = R * C
            if rc2_mode.value == 'Laden':
                Uc = U0 * (1 - math.exp(-t / tau))
                Ur = U0 - Uc
                I  = (U0 / R) * math.exp(-t / tau)
            else:
                Uc = U0 * math.exp(-t / tau)
                Ur = U0 - Uc
                I  = (U0 / R) * math.exp(-t / tau)
            print(f"τ     = {fmt(tau, 's')}")
            print(f"t/τ   = {t/tau:.4g}")
            print(f"U_C   = {fmt(Uc, 'V')}  ({Uc/U0*100:.2f}% von U₀)")
            print(f"U_R   = {fmt(Ur, 'V')}")
            print(f"I     = {fmt(I, 'A')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_rc2.on_click(calc_rc2)
rc2_mode.observe(calc_rc2, names='value')
for f in [rc2_R, rc2_C, rc2_U0, rc2_t]:
    f.observe(calc_rc2, names='value')
display(widgets.VBox([rc2_mode, rc2_R, rc2_C, rc2_U0, rc2_t, btn_rc2, out_rc2]))

---
## 2.5 Kondensator an Wechselspannung – Kapazitiver Widerstand

An Wechselspannung wirkt der Kondensator wie ein **frequenzabhängiger Widerstand**:

$$X_C = \frac{1}{2\pi \cdot f \cdot C} = \frac{1}{\omega \cdot C}$$

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Kapazitiver Widerstand | $X_C$ | Ω |
| Frequenz | $f$ | Hz |
| Kreisfrequenz | $\omega = 2\pi f$ | rad/s |
| Kapazität | $C$ | F |

> **Wichtig:** Bei höherer Frequenz → kleiner $X_C$ → Kondensator \"leitet besser\"
> Bei Gleichspannung (f = 0) → $X_C = \infty$ → Kondensator sperrt!

**Phasenverschiebung:** Der Strom **eilt** der Spannung um **90°** voraus.
$$I = \frac{U}{X_C}$$

In [ ]:
# ============================================================
# RECHNER 2.5: Kapazitiver Widerstand X_C = 1 / (2π·f·C)
# ============================================================
display(HTML("<h3>🔢 Rechner 2.5 – Kapazitiver Widerstand X_C</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

xc_Xc = make_input('X_C [Ω] =')
xc_f  = make_input('f [Hz]  =', 'z.B. 50, 1k, 10k')
xc_C  = make_input('C [F]   =', 'z.B. 100n, 4.7u')
xc_U  = make_input('U [V]   =', 'optional → I berechnen')
btn_xc = make_button()
out_xc = widgets.Output()

def calc_xc(_=None):
    out_xc.clear_output()
    with out_xc:
        try:
            Xc = p(xc_Xc.value); f = p(xc_f.value); C = p(xc_C.value)
            Uv = p(xc_U.value)
            vals = {'X_C': Xc, 'f': f, 'C': C}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld (X_C / f / C) leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'X_C': res = 1 / (2 * math.pi * f * C);  print(f"X_C = {fmt(res, 'Ω')}")
            elif m == 'f':   res = 1 / (2 * math.pi * Xc * C); print(f"f   = {fmt(res, 'Hz')}")
            else:            res = 1 / (2 * math.pi * f * Xc); print(f"C   = {fmt(res, 'F')}")
            Xc_val = Xc if Xc else res if m == 'X_C' else None
            if Uv and Xc_val:
                print(f"   I   = U/X_C = {fmt(Uv / Xc_val, 'A')}")
            f_val = f if f else res if m == 'f' else None
            C_val = C if C else res if m == 'C' else None
            if f_val and C_val:
                omega = 2 * math.pi * f_val
                print(f"   ω   = 2π·f = {fmt(omega, 'rad/s')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_xc.on_click(calc_xc)
for f_w in [xc_Xc, xc_f, xc_C, xc_U]:
    f_w.observe(calc_xc, names='value')
display(widgets.VBox([xc_Xc, xc_f, xc_C, xc_U, btn_xc, out_xc]))

---
## 2.6 RC-Reihenschaltung an Wechselspannung – Impedanz

$$Z = \sqrt{R^2 + X_C^2}$$

$$\varphi = -\arctan\!\left(\frac{X_C}{R}\right)$$

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Impedanz (Scheinwiderstand) | $Z$ | Ω |
| Wirkwiderstand | $R$ | Ω |
| Kapazitiver Widerstand | $X_C$ | Ω |
| Phasenwinkel | $\varphi$ | ° |

**Spannungen:**
$$U_{ges} = \sqrt{U_R^2 + U_C^2}$$

> ⚠️ Spannungen dürfen **nicht** direkt addiert werden – sie sind phasenverschoben!

In [ ]:
# ============================================================
# RECHNER 2.6: RC-Reihenschaltung Impedanz Z = √(R² + Xc²)
# ============================================================
display(HTML("<h3>🔢 Rechner 2.6 – RC-Reihenschaltung an Wechselspannung</h3>"
             "<i>R, f und C eingeben – alle Grössen werden berechnet</i>"))

rc_wR  = make_input('R [Ω]  =')
rc_wf  = make_input('f [Hz]  =', 'z.B. 50, 1k')
rc_wC  = make_input('C [F]   =', 'z.B. 100n')
rc_wU  = make_input('U_ges [V]=', 'optional')
btn_rcw = make_button()
out_rcw = widgets.Output()

def calc_rcw(_=None):
    out_rcw.clear_output()
    with out_rcw:
        try:
            R = p(rc_wR.value); f = p(rc_wf.value)
            C = p(rc_wC.value); U = p(rc_wU.value)
            if any(x is None for x in [R, f, C]):
                print("ℹ️ R, f und C eingeben"); return
            Xc = 1 / (2 * math.pi * f * C)
            Z  = math.sqrt(R**2 + Xc**2)
            phi = -math.degrees(math.atan(Xc / R))
            print(f"X_C  = {fmt(Xc, 'Ω')}")
            print(f"Z    = {fmt(Z, 'Ω')}")
            print(f"φ    = {phi:.2f}°  (Strom eilt Spannung vor)")
            if U:
                I = U / Z
                Ur = I * R
                Uc = I * Xc
                print(f"   I    = {fmt(I, 'A')}")
                print(f"   U_R  = {fmt(Ur, 'V')}")
                print(f"   U_C  = {fmt(Uc, 'V')}")
                print(f"   Probe: √(U_R²+U_C²) = {fmt(math.sqrt(Ur**2+Uc**2), 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_rcw.on_click(calc_rcw)
for f_w in [rc_wR, rc_wf, rc_wC, rc_wU]:
    f_w.observe(calc_rcw, names='value')
display(widgets.VBox([rc_wR, rc_wf, rc_wC, rc_wU, btn_rcw, out_rcw]))

---
## 2.7 Elektrisches Feld

Das **elektrische Feld** beschreibt den Einfluss einer Ladung auf den umgebenden Raum.

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Elektrische Feldstärke | $E$ | V/m |
| Elektrische Flussdichte | $D$ | C/m² |
| Spannung | $U$ | V |
| Plattenabstand | $d$ | m |
| Permittivität | $\varepsilon = \varepsilon_0 \cdot \varepsilon_r$ | F/m |

### Feldstärke im Plattenkondensator (homogenes Feld)
$$E = \frac{U}{d} \qquad [V/m]$$

### Elektrische Flussdichte
$$D = \varepsilon_0 \cdot \varepsilon_r \cdot E = \frac{Q}{A} \qquad [C/m^2]$$

### Zusammenhang
$$D = \varepsilon \cdot E \qquad C = \varepsilon \cdot \frac{A}{d}$$

> Die Feldstärke $E$ gibt an wie stark das Feld ist (unabhängig vom Material).
> Die Flussdichte $D$ berücksichtigt das Dielektrikum.

In [ ]:
# ============================================================
# RECHNER 2.7: Elektrisches Feld E und D
# ============================================================
display(HTML("<h3>🔢 Rechner 2.7 – Elektrisches Feld</h3>"
             "<i>Ein Feld leer lassen → wird berechnen</i>"))

EPS0 = 8.854e-12

ef_mat = widgets.Dropdown(
    options=[('-- Dielektrikum --', None), ('Luft/Vakuum', 1.0),
             ('Papier', 2.2), ('Polyester', 3.2), ('Glas', 6.0), ('Keramik', 100.0)],
    description='Material:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

ef_E   = make_input('E [V/m]  =')
ef_U   = make_input('U [V]    =')
ef_d   = make_input('d [m]    =', 'Plattenabstand')
ef_D   = make_input('D [C/m²] =', 'optional')
ef_er  = make_input('ε_r      =', 'für D-Berechnung')
btn_ef = make_button()
out_ef = widgets.Output()

def set_er_ef(change):
    if change['new'] is not None:
        ef_er.value = str(change['new'])
ef_mat.observe(set_er_ef, names='value')

def calc_ef(_=None):
    out_ef.clear_output()
    with out_ef:
        try:
            E  = p(ef_E.value); U = p(ef_U.value)
            d  = p(ef_d.value); er = p(ef_er.value)

            vals = {'E': E, 'U': U, 'd': d}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld (E, U oder d) leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'E': res = U / d;  print(f"  E = U/d = {fmt(res, 'V/m')}")
            elif m == 'U': res = E * d;  print(f"  U = E·d = {fmt(res, 'V')}")
            else:          res = U / E;  print(f"  d = U/E = {fmt(res, 'm')}")

            Ev = E if E else res if m == 'E' else None
            if Ev and er:
                D = EPS0 * er * Ev
                print(f"  D = ε₀·εᵣ·E = {D:.4e} C/m²")
                # Durchschlagsfeldstärke Hinweis
            if Ev:
                if Ev > 3e6:
                    print(f"  ⚠️ E > 3 MV/m – Durchschlagsgrenze Luft überschritten!")
        except Exception as e:
            print("❌ Fehler:", e)

btn_ef.on_click(calc_ef)
for fw in [ef_E, ef_U, ef_d, ef_er]:
    fw.observe(calc_ef, names='value')
display(widgets.VBox([ef_mat, ef_E, ef_U, ef_d, ef_er, btn_ef, out_ef]))

---
# Kapitel 3: Spulen (Induktivitäten)

---
## 3.1 Grundlagen der Spule

Eine **Spule** speichert elektrische Energie im **magnetischen Feld**.

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Induktivität | $L$ | Henry [H] |
| Spannung | $U_L$ | Volt [V] |
| Strom | $I$ | Ampere [A] |
| Magnetischer Fluss | $\Phi$ | Weber [Wb] |

### Grundformel – Selbstinduktion
$$U_L = L \cdot \frac{\Delta I}{\Delta t}$$

Die Spule **widersetzt sich jeder Stromänderung**.

### Energie in der Spule
$$W = \frac{1}{2} \cdot L \cdot I^2$$

In [ ]:
# ============================================================
# RECHNER 3.1: Selbstinduktionsspannung  U_L = L · ΔI/Δt
# ============================================================
display(HTML("<h3>🔢 Rechner 3.1 – Selbstinduktionsspannung</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

ul_UL  = make_input('U_L [V]    =')
ul_L   = make_input('L [H]      =', 'z.B. 10m, 4.7u')
ul_dI  = make_input('ΔI [A]     =', 'Stromänderung')
ul_dt  = make_input('Δt [s]     =', 'z.B. 1m = 1ms')
btn_ul = make_button()
out_ul = widgets.Output()

def calc_ul(_=None):
    out_ul.clear_output()
    with out_ul:
        try:
            UL = p(ul_UL.value); L  = p(ul_L.value)
            dI = p(ul_dI.value); dt = p(ul_dt.value)
            vals = {'U_L': UL, 'L': L, 'ΔI': dI, 'Δt': dt}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'U_L': res = L * dI / dt;  print(f"U_L = L·ΔI/Δt = {fmt(res, 'V')}")
            elif m == 'L':   res = UL * dt / dI; print(f"L   = U_L·Δt/ΔI = {fmt(res, 'H')}")
            elif m == 'ΔI':  res = UL * dt / L;  print(f"ΔI  = U_L·Δt/L = {fmt(res, 'A')}")
            else:            res = L * dI / UL;  print(f"Δt  = L·ΔI/U_L = {fmt(res, 's')}")
            # Energie wenn L und I bekannt
            Lv = L if L else (res if m == 'L' else None)
            Iv = dI if dI else (res if m == 'ΔI' else None)
            if Lv and Iv:
                print(f"   W = ½·L·I² = {fmt(0.5 * Lv * Iv**2, 'J')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_ul.on_click(calc_ul)
for f in [ul_UL, ul_L, ul_dI, ul_dt]:
    f.observe(calc_ul, names='value')
display(widgets.VBox([ul_UL, ul_L, ul_dI, ul_dt, btn_ul, out_ul]))

---
## 3.2 Induktivität einer Spule (Zylinderspule)

$$L = \mu_0 \cdot \mu_r \cdot \frac{N^2 \cdot A}{l}$$

| Grösse | Symbol | Wert / Einheit |
|--------|--------|----------------|
| Permeabilität Vakuum | $\mu_0$ | $4\pi \times 10^{-7}$ H/m |
| Relative Permeabilität | $\mu_r$ | dimensionslos |
| Windungszahl | $N$ | – |
| Querschnittsfläche | $A$ | m² |
| Spulenlänge | $l$ | m |

| Material | $\mu_r$ |
|----------|---------|
| Luft / Vakuum | 1 |
| Ferrit | 10–10'000 |
| Siliziumstahl | 1'000–10'000 |
| Mu-Metall | bis 100'000 |

In [ ]:
# ============================================================
# RECHNER 3.2: Induktivität Zylinderspule  L = µ0·µr·N²·A/l
# ============================================================
display(HTML("<h3>🔢 Rechner 3.2 – Induktivität Zylinderspule</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

MU0 = 4 * math.pi * 1e-7

kern_mat = widgets.Dropdown(
    options=[('-- Kernmaterial --', None), ('Luft/Vakuum', 1),
             ('Ferrit (mittel)', 1000), ('Siliziumstahl', 5000), ('Mu-Metall', 50000)],
    description='Kernmaterial:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

sp_L   = make_input('L [H]    =', 'leer = berechnen')
sp_mur = make_input('µ_r      =', 'z.B. 1 für Luft')
sp_N   = make_input('N (Windungen) =', 'Anzahl')
sp_A   = make_input('A [m²]   =', 'z.B. 0.001')
sp_l   = make_input('l [m]    =', 'Spulenlänge')
btn_sp = make_button()
out_sp = widgets.Output()

def set_mur(change):
    if change['new'] is not None:
        sp_mur.value = str(change['new'])
kern_mat.observe(set_mur, names='value')

def calc_sp(_=None):
    out_sp.clear_output()
    with out_sp:
        try:
            L   = p(sp_L.value)
            mur = p(sp_mur.value)
            N   = p(sp_N.value)
            A   = p(sp_A.value)
            l   = p(sp_l.value)
            vals = {'L': L, 'µ_r': mur, 'N': N, 'A': A, 'l': l}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'L':   res = MU0 * mur * N**2 * A / l;           print(f"L   = {fmt(res, 'H')}")
            elif m == 'µ_r': res = L * l / (MU0 * N**2 * A);           print(f"µ_r = {res:.4g}")
            elif m == 'N':   res = math.sqrt(L * l / (MU0 * mur * A)); print(f"N   = {res:.4g} Windungen")
            elif m == 'A':   res = L * l / (MU0 * mur * N**2);         print(f"A   = {fmt(res, 'm²')}")
            else:            res = MU0 * mur * N**2 * A / L;           print(f"l   = {fmt(res, 'm')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_sp.on_click(calc_sp)
for f in [sp_L, sp_mur, sp_N, sp_A, sp_l]:
    f.observe(calc_sp, names='value')
display(widgets.VBox([kern_mat, sp_L, sp_mur, sp_N, sp_A, sp_l, btn_sp, out_sp]))

---
## 3.3 Spulen in Reihe und Parallel

### Reihenschaltung (ohne gegenseitige Beeinflussung)
$$L_{ges} = L_1 + L_2 + L_3 + \ldots$$

### Parallelschaltung
$$\frac{1}{L_{ges}} = \frac{1}{L_1} + \frac{1}{L_2} + \frac{1}{L_3} + \ldots$$

> Gleiche Regeln wie bei Widerständen – aber umgekehrt zu Kondensatoren!

In [ ]:
# ============================================================
# RECHNER 3.3: Spulen Reihe / Parallel
# ============================================================
display(HTML("<h3>🔢 Rechner 3.3 – Reihe & Parallel</h3>"
             "<i>Bis 5 Spulen. Leere Felder werden ignoriert.</i>"))

l_toggle = widgets.ToggleButtons(
    options=['Reihenschaltung', 'Parallelschaltung'],
    description='Schaltung:', style={'description_width': '130px', 'button_width': '160px'}
)
l_fields = [make_input(f'L{i+1} [H] =', 'z.B. 10m, 4.7u') for i in range(5)]
btn_lp = make_button()
out_lp = widgets.Output()

def calc_lp(_=None):
    out_lp.clear_output()
    with out_lp:
        try:
            Ls = [p(f.value) for f in l_fields]
            Ls_v = [l for l in Ls if l is not None]
            if len(Ls_v) < 1:
                print("ℹ️ Mindestens 1 Spule eingeben"); return
            if l_toggle.value == 'Reihenschaltung':
                L_ges = sum(Ls_v)
                print(f"L_ges (Reihe)    = {fmt(L_ges, 'H')}")
            else:
                L_ges = 1 / sum(1/l for l in Ls_v)
                print(f"L_ges (Parallel) = {fmt(L_ges, 'H')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_lp.on_click(calc_lp)
l_toggle.observe(calc_lp, names='value')
for f in l_fields:
    f.observe(calc_lp, names='value')
display(widgets.VBox([l_toggle] + l_fields + [btn_lp, out_lp]))

---
## 3.4 RL-Glied – Zeitkonstante und Ein-/Ausschalten

$$\tau = \frac{L}{R}$$

| Zeit | Strom beim Einschalten | Strom beim Ausschalten |
|------|------------------------|------------------------|
| $1\tau$ | 63.2% von $I_{max}$ | 36.8% |
| $2\tau$ | 86.5% | 13.5% |
| $3\tau$ | 95.0% | 5.0% |
| $5\tau$ | 99.3% | 0.7% ≈ vollständig |

**Einschalten (Strom steigt):**
$$i(t) = I_{max} \cdot \left(1 - e^{-t/\tau}\right) \qquad I_{max} = \frac{U_0}{R}$$

**Ausschalten (Strom fällt):**
$$i(t) = I_{max} \cdot e^{-t/\tau}$$

**Spannung an der Spule:**
$$u_L(t) = U_0 \cdot e^{-t/\tau}$$

> ⚠️ Beim **Ausschalten** entsteht eine hohe Induktionsspannung! Deshalb werden Freilaufdioden verwendet.

In [ ]:
# ============================================================
# RECHNER 3.4a: Zeitkonstante τ = L/R
# ============================================================
display(HTML("<h3>🔢 Rechner 3.4a – Zeitkonstante τ = L/R</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

rl_L   = make_input('L [H]  =', 'z.B. 10m, 4.7u')
rl_R   = make_input('R [Ω]  =')
rl_tau = make_input('τ [s]  =')
btn_rlt = make_button()
out_rlt = widgets.Output()

def calc_rlt(_=None):
    out_rlt.clear_output()
    with out_rlt:
        try:
            L = p(rl_L.value); R = p(rl_R.value); T = p(rl_tau.value)
            vals = {'L': L, 'R': R, 'τ': T}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'τ': res = L / R;   print(f"τ = L/R = {fmt(res, 's')}")
            elif m == 'L': res = T * R;   print(f"L = τ·R = {fmt(res, 'H')}")
            else:          res = L / T;   print(f"R = L/τ = {fmt(res, 'Ω')}")
            tau_val = T if T else res
            print(f"\n   Stromtabelle (Einschalten):")
            for n in [1, 2, 3, 4, 5]:
                ein_p = (1 - math.exp(-n)) * 100
                aus_p = math.exp(-n) * 100
                print(f"   {n}τ = {fmt(n*tau_val,'s'):12s}  → Ein: {ein_p:.1f}%   Aus: {aus_p:.1f}%")
        except Exception as e:
            print("❌ Fehler:", e)

btn_rlt.on_click(calc_rlt)
for f in [rl_L, rl_R, rl_tau]:
    f.observe(calc_rlt, names='value')
display(widgets.VBox([rl_L, rl_R, rl_tau, btn_rlt, out_rlt]))

In [ ]:
# ============================================================
# RECHNER 3.4b: Strom / Spannung zu Zeitpunkt t im RL-Glied
# ============================================================
display(HTML("<h3>🔢 Rechner 3.4b – I und U_L zu Zeitpunkt t</h3>"
             "<i>Alle 4 Felder ausfüllen</i>"))

rl2_mode = widgets.ToggleButtons(
    options=['Einschalten', 'Ausschalten'],
    description='Vorgang:', style={'description_width': '130px', 'button_width': '120px'}
)
rl2_L  = make_input('L [H]   =', 'z.B. 10m')
rl2_R  = make_input('R [Ω]   =')
rl2_U0 = make_input('U₀ [V]  =', 'Quellspannung')
rl2_t  = make_input('t [s]   =', 'z.B. 5m = 5ms')
btn_rl2 = make_button()
out_rl2 = widgets.Output()

def calc_rl2(_=None):
    out_rl2.clear_output()
    with out_rl2:
        try:
            L  = p(rl2_L.value);  R  = p(rl2_R.value)
            U0 = p(rl2_U0.value); t  = p(rl2_t.value)
            if any(x is None for x in [L, R, U0, t]):
                print("ℹ️ Alle 4 Felder ausfüllen"); return
            tau   = L / R
            I_max = U0 / R
            if rl2_mode.value == 'Einschalten':
                I  = I_max * (1 - math.exp(-t / tau))
                UL = U0 * math.exp(-t / tau)
                UR = I * R
            else:
                I  = I_max * math.exp(-t / tau)
                UL = -U0 * math.exp(-t / tau)
                UR = I * R
            print(f"τ      = {fmt(tau, 's')}")
            print(f"I_max  = {fmt(I_max, 'A')}")
            print(f"t/τ    = {t/tau:.4g}")
            print(f"I      = {fmt(I, 'A')}  ({I/I_max*100:.2f}% von I_max)")
            print(f"U_L    = {fmt(UL, 'V')}")
            print(f"U_R    = {fmt(UR, 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_rl2.on_click(calc_rl2)
rl2_mode.observe(calc_rl2, names='value')
for f in [rl2_L, rl2_R, rl2_U0, rl2_t]:
    f.observe(calc_rl2, names='value')
display(widgets.VBox([rl2_mode, rl2_L, rl2_R, rl2_U0, rl2_t, btn_rl2, out_rl2]))

---
## 3.5 Spule an Wechselspannung – Induktiver Widerstand

$$X_L = 2\pi \cdot f \cdot L = \omega \cdot L$$

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Induktiver Widerstand | $X_L$ | Ω |
| Frequenz | $f$ | Hz |
| Kreisfrequenz | $\omega = 2\pi f$ | rad/s |
| Induktivität | $L$ | H |

> **Wichtig:** Bei höherer Frequenz → grösser $X_L$ → Spule sperrt besser
> Bei Gleichspannung (f = 0) → $X_L = 0$ → Spule ist nur ihr Wicklungswiderstand

**Phasenverschiebung:** Der Strom **eilt** der Spannung um **90° nach**.
$$I = \frac{U}{X_L}$$

In [ ]:
# ============================================================
# RECHNER 3.5: Induktiver Widerstand  X_L = 2π·f·L
# ============================================================
display(HTML("<h3>🔢 Rechner 3.5 – Induktiver Widerstand X_L</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

xl_XL = make_input('X_L [Ω] =')
xl_f  = make_input('f [Hz]  =', 'z.B. 50, 1k, 10k')
xl_L  = make_input('L [H]   =', 'z.B. 10m, 4.7u')
xl_U  = make_input('U [V]   =', 'optional → I berechnen')
btn_xl = make_button()
out_xl = widgets.Output()

def calc_xl(_=None):
    out_xl.clear_output()
    with out_xl:
        try:
            XL = p(xl_XL.value); f = p(xl_f.value)
            L  = p(xl_L.value);  Uv = p(xl_U.value)
            vals = {'X_L': XL, 'f': f, 'L': L}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld (X_L / f / L) leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'X_L': res = 2 * math.pi * f * L;    print(f"X_L = {fmt(res, 'Ω')}")
            elif m == 'f':   res = XL / (2 * math.pi * L); print(f"f   = {fmt(res, 'Hz')}")
            else:            res = XL / (2 * math.pi * f); print(f"L   = {fmt(res, 'H')}")
            XL_val = XL if XL else res if m == 'X_L' else None
            if Uv and XL_val:
                print(f"   I   = U/X_L = {fmt(Uv / XL_val, 'A')}")
            f_val = f if f else res if m == 'f' else None
            L_val = L if L else res if m == 'L' else None
            if f_val:
                print(f"   ω   = 2π·f = {fmt(2 * math.pi * f_val, 'rad/s')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_xl.on_click(calc_xl)
for fw in [xl_XL, xl_f, xl_L, xl_U]:
    fw.observe(calc_xl, names='value')
display(widgets.VBox([xl_XL, xl_f, xl_L, xl_U, btn_xl, out_xl]))

---
## 3.6 RL-Reihenschaltung an Wechselspannung – Impedanz

$$Z = \sqrt{R^2 + X_L^2}$$

$$\varphi = +\arctan\!\left(\frac{X_L}{R}\right)$$

**Spannungen (nicht direkt addierbar!):**
$$U_{ges} = \sqrt{U_R^2 + U_L^2}$$

| Vergleich | RC-Glied | RL-Glied |
|-----------|----------|----------|
| Phasenverschiebung | $\varphi < 0°$ (I eilt vor) | $\varphi > 0°$ (I eilt nach) |
| Widerstand steigt mit f | Nein | Ja |
| Widerstand fällt mit f | Ja | Nein |

In [ ]:
# ============================================================
# RECHNER 3.6: RL-Reihenschaltung Impedanz Z = √(R² + XL²)
# ============================================================
display(HTML("<h3>🔢 Rechner 3.6 – RL-Reihenschaltung an Wechselspannung</h3>"
             "<i>R, f und L eingeben – alle Grössen werden berechnet</i>"))

rl_wR  = make_input('R [Ω]   =')
rl_wf  = make_input('f [Hz]  =', 'z.B. 50, 1k')
rl_wL  = make_input('L [H]   =', 'z.B. 10m')
rl_wU  = make_input('U_ges [V]=', 'optional')
btn_rlw = make_button()
out_rlw = widgets.Output()

def calc_rlw(_=None):
    out_rlw.clear_output()
    with out_rlw:
        try:
            R = p(rl_wR.value); f = p(rl_wf.value)
            L = p(rl_wL.value); U = p(rl_wU.value)
            if any(x is None for x in [R, f, L]):
                print("ℹ️ R, f und L eingeben"); return
            XL  = 2 * math.pi * f * L
            Z   = math.sqrt(R**2 + XL**2)
            phi = math.degrees(math.atan(XL / R))
            print(f"X_L  = {fmt(XL, 'Ω')}")
            print(f"Z    = {fmt(Z, 'Ω')}")
            print(f"φ    = +{phi:.2f}°  (Strom eilt Spannung nach)")
            if U:
                I  = U / Z
                Ur = I * R
                Ul = I * XL
                print(f"   I    = {fmt(I, 'A')}")
                print(f"   U_R  = {fmt(Ur, 'V')}")
                print(f"   U_L  = {fmt(Ul, 'V')}")
                print(f"   Probe: √(U_R²+U_L²) = {fmt(math.sqrt(Ur**2+Ul**2), 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_rlw.on_click(calc_rlw)
for fw in [rl_wR, rl_wf, rl_wL, rl_wU]:
    fw.observe(calc_rlw, names='value')
display(widgets.VBox([rl_wR, rl_wf, rl_wL, rl_wU, btn_rlw, out_rlw]))

---
## 3.7 Magnetisches Feld

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Magnetische Feldstärke | $H$ | A/m |
| Magnetische Flussdichte | $B$ | Tesla [T] |
| Magnetischer Fluss | $\Phi$ | Weber [Wb] |
| Durchflutung | $\Theta$ | A (Amperewindungen) |
| Permeabilität | $\mu = \mu_0 \cdot \mu_r$ | H/m |

### Durchflutung
$$\Theta = N \cdot I \qquad [A]$$

### Feldstärke in einer Spule
$$H = \frac{\Theta}{l} = \frac{N \cdot I}{l} \qquad [A/m]$$

### Flussdichte
$$B = \mu_0 \cdot \mu_r \cdot H \qquad [T]$$

### Magnetischer Fluss
$$\Phi = B \cdot A \qquad [Wb] = [V \cdot s]$$

### Induktionsgesetz
$$U_{ind} = N \cdot \frac{\Delta \Phi}{\Delta t} \qquad [V]$$

In [ ]:
# ============================================================
# RECHNER 3.7: Magnetisches Feld – H, B, Φ, Θ
# ============================================================
display(HTML("<h3>🔢 Rechner 3.7 – Magnetisches Feld</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

MU0 = 4 * math.pi * 1e-7

mf_kern = widgets.Dropdown(
    options=[('-- Kernmaterial --', None), ('Luft/Vakuum', 1),
             ('Ferrit (mittel)', 1000), ('Siliziumstahl', 5000), ('Mu-Metall', 50000)],
    description='Kernmaterial:', style={'description_width': '140px'},
    layout=widgets.Layout(width='340px')
)

mf_N   = make_input('N (Windungen) =')
mf_I   = make_input('I [A]         =')
mf_l   = make_input('l [m]         =', 'Magnetkreislänge')
mf_A   = make_input('A [m²]        =', 'Kernquerschnitt')
mf_mur = make_input('µ_r           =', 'rel. Permeabilität')
btn_mf = make_button()
out_mf = widgets.Output()

def set_mur_mf(change):
    if change['new'] is not None:
        mf_mur.value = str(change['new'])
mf_kern.observe(set_mur_mf, names='value')

def calc_mf(_=None):
    out_mf.clear_output()
    with out_mf:
        try:
            N   = p(mf_N.value);   I   = p(mf_I.value)
            l   = p(mf_l.value);   A   = p(mf_A.value)
            mur = p(mf_mur.value)

            if any(x is None for x in [N, I, l]):
                print("ℹ️ N, I und l eingeben"); return

            Theta = N * I
            H     = Theta / l
            print(f"  Θ = N·I  = {fmt(Theta, 'A')}  (Amperewindungen)")
            print(f"  H = Θ/l  = {fmt(H, 'A/m')}")

            if mur:
                B   = MU0 * mur * H
                print(f"  B = µ₀·µᵣ·H = {B:.4e} T")
                if A:
                    Phi = B * A
                    print(f"  Φ = B·A  = {Phi:.4e} Wb")
                # Sättigung prüfen
                if mur > 1 and B > 1.5:
                    print(f"  ⚠️ B > 1.5 T – Kernsättigung möglich!")
        except Exception as e:
            print("❌ Fehler:", e)

btn_mf.on_click(calc_mf)
for fw in [mf_N, mf_I, mf_l, mf_A, mf_mur]:
    fw.observe(calc_mf, names='value')
display(widgets.VBox([mf_kern, mf_N, mf_I, mf_l, mf_A, mf_mur, btn_mf, out_mf]))

---
# Kapitel 4: RLC-Schwingkreise

---
## 4.1 Grundlagen des Schwingkreises

Ein **Schwingkreis** besteht aus einer Spule $L$ und einem Kondensator $C$.
Energie pendelt zwischen magnetischem Feld (Spule) und elektrischem Feld (Kondensator).

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Resonanzfrequenz | $f_0$ | Hz |
| Kreisfrequenz | $\omega_0$ | rad/s |
| Induktivität | $L$ | H |
| Kapazität | $C$ | F |
| Widerstand | $R$ | Ω |

### Resonanzfrequenz
$$f_0 = \frac{1}{2\pi \cdot \sqrt{L \cdot C}} \qquad \omega_0 = \frac{1}{\sqrt{L \cdot C}}$$

Bei Resonanz gilt: $X_L = X_C$ → die Blindwiderstände heben sich auf!

In [ ]:
# ============================================================
# RECHNER 4.1: Resonanzfrequenz  f0 = 1 / (2π·√(L·C))
# ============================================================
display(HTML("<h3>🔢 Rechner 4.1 – Resonanzfrequenz</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

res_f  = make_input('f₀ [Hz] =')
res_L  = make_input('L [H]   =', 'z.B. 10m, 4.7u')
res_C  = make_input('C [F]   =', 'z.B. 100n, 4.7u')
btn_res = make_button()
out_res = widgets.Output()

def calc_res(_=None):
    out_res.clear_output()
    with out_res:
        try:
            f0 = p(res_f.value); L = p(res_L.value); C = p(res_C.value)
            vals = {'f₀': f0, 'L': L, 'C': C}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'f₀': res = 1 / (2*math.pi*math.sqrt(L*C)); print(f"f₀ = {fmt(res, 'Hz')}")
            elif m == 'L':  res = 1 / (4*math.pi**2 * f0**2 * C); print(f"L  = {fmt(res, 'H')}")
            else:           res = 1 / (4*math.pi**2 * f0**2 * L); print(f"C  = {fmt(res, 'F')}")
            f_val = f0 if f0 else res if m == 'f₀' else None
            L_val = L  if L  else res if m == 'L'  else None
            C_val = C  if C  else res if m == 'C'  else None
            if f_val:
                omega = 2 * math.pi * f_val
                print(f"   ω₀  = {fmt(omega, 'rad/s')}")
            if L_val and C_val:
                XL = 2 * math.pi * f_val * L_val if f_val else None
                if XL: print(f"   X_L = X_C = {fmt(XL, 'Ω')}  (bei Resonanz)")
        except Exception as e:
            print("❌ Fehler:", e)

btn_res.on_click(calc_res)
for fw in [res_f, res_L, res_C]:
    fw.observe(calc_res, names='value')
display(widgets.VBox([res_f, res_L, res_C, btn_res, out_res]))

---
## 4.2 Reihenschwingkreis

Im **Reihenschwingkreis** liegen R, L und C in Reihe.

$$Z = \sqrt{R^2 + (X_L - X_C)^2}$$

Bei Resonanz: $X_L = X_C$ → $Z = R$ (minimaler Widerstand → maximaler Strom!)

### Güte (Qualitätsfaktor)
$$Q = \frac{1}{R} \cdot \sqrt{\frac{L}{C}} = \frac{f_0}{\Delta f} = \frac{X_L}{R} = \frac{X_C}{R}$$

- Hohe Güte $Q$ → schmales, scharfes Resonanzverhalten
- Typische Werte: $Q = 10 \ldots 200$

### Bandbreite
Frequenzbereich zwischen den beiden −3dB Punkten (halbe Leistung).
$$\Delta f = \frac{f_0}{Q} \qquad [Hz]$$

### Grenzfrequenzen
$$f_u = f_0 - \frac{\Delta f}{2} \qquad f_o = f_0 + \frac{\Delta f}{2}$$

### Resonanzüberhöhung
Bei Resonanz sind $U_L$ und $U_C$ jeweils $Q$-mal grösser als $U_{ges}$!
$$U_L = U_C = Q \cdot U_{ges}$$

In [ ]:
# ============================================================
# RECHNER 4.2: Reihenschwingkreis – Z, Q, Δf
# ============================================================
display(HTML("<h3>🔢 Rechner 4.2 – Reihenschwingkreis</h3>"
             "<i>R, L, C und Frequenz eingeben</i>"))

sr_R  = make_input('R [Ω]   =')
sr_L  = make_input('L [H]   =', 'z.B. 10m')
sr_C  = make_input('C [F]   =', 'z.B. 100n')
sr_f  = make_input('f [Hz]  =', 'Betriebsfreq. (leer = f₀)')
sr_U  = make_input('U_ges [V]=', 'optional')
btn_sr = make_button()
out_sr = widgets.Output()

def calc_sr(_=None):
    out_sr.clear_output()
    with out_sr:
        try:
            R = p(sr_R.value); L = p(sr_L.value)
            C = p(sr_C.value); U = p(sr_U.value)
            f_in = p(sr_f.value)
            if any(x is None for x in [R, L, C]):
                print("ℹ️ R, L und C eingeben"); return

            f0    = 1 / (2*math.pi*math.sqrt(L*C))
            omega0 = 2*math.pi*f0
            f     = f_in if f_in else f0
            omega = 2*math.pi*f
            XL    = omega * L
            XC    = 1 / (omega * C)
            Z     = math.sqrt(R**2 + (XL - XC)**2)
            Q     = (1/R) * math.sqrt(L/C)
            df    = f0 / Q
            f_u   = f0 - df/2
            f_o   = f0 + df/2

            print(f"--- Resonanz ---")
            print(f"f₀    = {fmt(f0, 'Hz')}")
            print(f"ω₀    = {fmt(omega0, 'rad/s')}")
            print(f"Q     = {Q:.4g}")
            print(f"Δf    = {fmt(df, 'Hz')}")
            print(f"f_u   = {fmt(f_u, 'Hz')}")
            print(f"f_o   = {fmt(f_o, 'Hz')}")
            print(f"\n--- Bei f = {fmt(f, 'Hz')} ---")
            print(f"X_L   = {fmt(XL, 'Ω')}")
            print(f"X_C   = {fmt(XC, 'Ω')}")
            print(f"Z     = {fmt(Z, 'Ω')}")
            if Z == R:
                print(f"   → Resonanz! Z = R (minimal)")
            elif XL > XC:
                print(f"   → Induktiv (f > f₀)")
            else:
                print(f"   → Kapazitiv (f < f₀)")
            if U:
                I  = U / Z
                UR = I * R
                UL = I * XL
                UC = I * XC
                print(f"I     = {fmt(I, 'A')}")
                print(f"U_R   = {fmt(UR, 'V')}")
                print(f"U_L   = {fmt(UL, 'V')}")
                print(f"U_C   = {fmt(UC, 'V')}")
                if f_in is None:
                    print(f"\n   Resonanzüberhöhung:")
                    print(f"   U_L = U_C = Q·U = {fmt(Q*U, 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_sr.on_click(calc_sr)
for fw in [sr_R, sr_L, sr_C, sr_f, sr_U]:
    fw.observe(calc_sr, names='value')
display(widgets.VBox([sr_R, sr_L, sr_C, sr_f, sr_U, btn_sr, out_sr]))

---
## 4.3 Parallelschwingkreis

Im **Parallelschwingkreis** liegen L und C parallel (R ist der Verlustwiderstand der Spule).

Bei Resonanz: $Z = Z_{max}$ (maximaler Widerstand → minimaler Strom aus der Quelle!)

### Resonanzfrequenz (mit Verlustwiderstand)
$$f_0 = \frac{1}{2\pi} \cdot \sqrt{\frac{1}{LC} - \frac{R^2}{L^2}}$$

Für $Q \gg 1$ (verlustarm) vereinfacht sich dies zu:
$$f_0 \approx \frac{1}{2\pi\sqrt{LC}}$$

### Güte
$$Q = \frac{\omega_0 \cdot L}{R} = \frac{X_L}{R}$$

### Impedanz bei Resonanz (Parallelwiderstand)
$$R_p = Q^2 \cdot R = \frac{L}{R \cdot C}$$

### Bandbreite
Frequenzbereich zwischen den beiden −3dB Punkten (halbe Leistung).
$$\Delta f = \frac{f_0}{Q}$$

In [ ]:
# ============================================================
# RECHNER 4.3: Parallelschwingkreis – f0, Q, Rp, Δf
# ============================================================
display(HTML("<h3>🔢 Rechner 4.3 – Parallelschwingkreis</h3>"
             "<i>R (Spulenwiderstand), L und C eingeben</i>"))

par_R  = make_input('R [Ω]   =', 'Verlustwiderstand Spule')
par_L  = make_input('L [H]   =', 'z.B. 10m')
par_C  = make_input('C [F]   =', 'z.B. 100n')
par_U  = make_input('U [V]   =', 'optional')
btn_par = make_button()
out_par = widgets.Output()

def calc_par(_=None):
    out_par.clear_output()
    with out_par:
        try:
            R = p(par_R.value); L = p(par_L.value)
            C = p(par_C.value); U = p(par_U.value)
            if any(x is None for x in [R, L, C]):
                print("ℹ️ R, L und C eingeben"); return

            omega0_ideal = 1 / math.sqrt(L*C)
            # Exakte Formel mit Verlust
            radicand = 1/(L*C) - (R/L)**2
            if radicand <= 0:
                print("⚠️ R zu gross – kein Schwingen möglich (R ≥ √(L/C))")
                return
            omega0 = math.sqrt(radicand)
            f0     = omega0 / (2*math.pi)
            f0_id  = omega0_ideal / (2*math.pi)
            XL     = omega0_ideal * L
            Q      = XL / R
            Rp     = Q**2 * R
            df     = f0 / Q
            f_u    = f0 - df/2
            f_o    = f0 + df/2

            print(f"--- Parallelschwingkreis ---")
            print(f"f₀ (exakt)  = {fmt(f0, 'Hz')}")
            print(f"f₀ (ideal)  = {fmt(f0_id, 'Hz')}")
            print(f"Q           = {Q:.4g}")
            print(f"R_p (Z_max) = {fmt(Rp, 'Ω')}")
            print(f"Δf          = {fmt(df, 'Hz')}")
            print(f"f_u         = {fmt(f_u, 'Hz')}")
            print(f"f_o         = {fmt(f_o, 'Hz')}")
            if Q < 10:
                print(f"\n   ⚠️ Q = {Q:.2f} < 10 → Näherung f₀ ≈ 1/(2π√LC) ungenau!")
            if U:
                I_ges = U / Rp
                print(f"\n   Bei U = {fmt(U, 'V')}:")
                print(f"   I_ges (Quelle) = {fmt(I_ges, 'A')}  (minimal bei Resonanz)")
                print(f"   I_L = I_C      = {fmt(Q * I_ges, 'A')}  (Kreisström!)")
        except Exception as e:
            print("❌ Fehler:", e)

btn_par.on_click(calc_par)
for fw in [par_R, par_L, par_C, par_U]:
    fw.observe(calc_par, names='value')
display(widgets.VBox([par_R, par_L, par_C, par_U, btn_par, out_par]))

---
## 4.4 Güte und Bandbreite – Vergleich

$$Q = \frac{f_0}{\Delta f} \qquad \Delta f = f_o - f_u$$

| Güte $Q$ | Charakter | Typische Anwendung |
|----------|-----------|--------------------|
| $Q < 1$ | Überdämpft – kein Schwingen | Dämpfungsglieder |
| $Q = 1 \ldots 5$ | Breitbandig | Audio-Filter |
| $Q = 10 \ldots 50$ | Mittel | ZF-Filter, Empfänger |
| $Q > 100$ | Schmalbandig | Quarzoszillatoren |

### Zusammenhang Güte – Dämpfung
$$d = \frac{1}{Q} \qquad \text{(Dämpfungsgrad)}$$

| $d$ | $Q$ | Verhalten |
|-----|-----|-----------|
| $d > 1$ | $Q < 0.5$ | Überdämpft |
| $d = 1$ | $Q = 0.5$ | Aperiodischer Grenzfall |
| $d < 1$ | $Q > 0.5$ | Unterdämpft (schwingt) |

In [ ]:
# ============================================================
# RECHNER 4.4: Güte / Bandbreite – Umrechner
# ============================================================
display(HTML("<h3>🔢 Rechner 4.4 – Güte und Bandbreite</h3>"
             "<i>Zwei von drei Feldern eingeben</i>"))

qb_f0 = make_input('f₀ [Hz] =', 'Resonanzfrequenz')
qb_Q  = make_input('Q       =', 'Güte')
qb_df = make_input('Δf [Hz] =', 'Bandbreite')
btn_qb = make_button()
out_qb = widgets.Output()

def calc_qb(_=None):
    out_qb.clear_output()
    with out_qb:
        try:
            f0 = p(qb_f0.value); Q = p(qb_Q.value); df = p(qb_df.value)
            vals = {'f₀': f0, 'Q': Q, 'Δf': df}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'Δf': res = f0 / Q;   label = 'Δf'; unit = 'Hz'
            elif m == 'Q':  res = f0 / df;  label = 'Q';  unit = ''
            else:           res = Q * df;   label = 'f₀'; unit = 'Hz'

            f0v  = f0  if f0  else (res if m == 'f₀' else None)
            Qv   = Q   if Q   else (res if m == 'Q'  else None)
            dfv  = df  if df  else (res if m == 'Δf' else None)

            print(f"{label} = {fmt(res, unit)}" if unit else f"{label} = {res:.4g}")
            if f0v and dfv:
                fu = f0v - dfv/2
                fo = f0v + dfv/2
                print(f"   f_u = {fmt(fu, 'Hz')}")
                print(f"   f_o = {fmt(fo, 'Hz')}")
            if Qv:
                d = 1/Qv
                print(f"   d   = 1/Q = {d:.4g}")
                if   Qv < 0.5:  print(f"   → Überdämpft")
                elif Qv == 0.5: print(f"   → Aperiodischer Grenzfall")
                elif Qv < 10:   print(f"   → Unterdämpft, breitbandig")
                elif Qv < 100:  print(f"   → Mittlere Güte")
                else:           print(f"   → Hohe Güte, schmalbandig")
        except Exception as e:
            print("❌ Fehler:", e)

btn_qb.on_click(calc_qb)
for fw in [qb_f0, qb_Q, qb_df]:
    fw.observe(calc_qb, names='value')
display(widgets.VBox([qb_f0, qb_Q, qb_df, btn_qb, out_qb]))

---
# Kapitel 5: Dezibel (dB)

---
## 5.1 Grundlagen

Das **Dezibel** ist ein logarithmisches Mass für Verhältnisse.
Es wird verwendet um grosse Wertebereiche kompakt darzustellen.

### Leistungs-Dezibel
$$L_P = 10 \cdot \lg\left(\frac{P_2}{P_1}\right) \quad [dB]$$

### Spannungs- und Strom-Dezibel
$$L_U = 20 \cdot \lg\left(\frac{U_2}{U_1}\right) \quad [dB]$$
$$L_I = 20 \cdot \lg\left(\frac{I_2}{I_1}\right) \quad [dB]$$

> Der Faktor 20 (statt 10) kommt daher, dass $P \propto U^2$ bzw. $P \propto I^2$.

### Vorzeichen
| Ergebnis | Bedeutung |
|----------|-----------|
| $> 0\,dB$ | Verstärkung |
| $= 0\,dB$ | Keine Änderung ($U_2 = U_1$) |
| $< 0\,dB$ | Dämpfung |

In [ ]:
# ============================================================
# RECHNER 5.1: Dezibel Grundrechner  (Spannung / Strom / Leistung)
# ============================================================
display(HTML("<h3>🔢 Rechner 5.1 – Dezibel Grundrechner</h3>"
             "<i>Zwei Werte eingeben → dB, oder dB + einen Wert → anderen berechnen</i>"))

db_mode = widgets.ToggleButtons(
    options=['Spannung (20·lg)', 'Strom (20·lg)', 'Leistung (10·lg)'],
    description='Grösse:', style={'description_width': '130px', 'button_width': '140px'}
)
db_x1  = make_input('Eingang (x₁) =', 'z.B. 1, 100m')
db_x2  = make_input('Ausgang (x₂) =', 'leer = aus dB berechnen')
db_dB  = make_input('dB           =', 'leer = berechnen')
btn_db = make_button()
out_db = widgets.Output()

def calc_db(_=None):
    out_db.clear_output()
    with out_db:
        try:
            x1 = p(db_x1.value); x2 = p(db_x2.value); dB = p(db_dB.value)
            factor = 10 if db_mode.value == 'Leistung (10·lg)' else 20

            if x1 is not None and x2 is not None:
                res = factor * math.log10(x2 / x1)
                unit = 'V' if 'Spannung' in db_mode.value else ('A' if 'Strom' in db_mode.value else 'W')
                print(f"  dB  = {res:.4g} dB")
                if res > 0:   print(f"  → Verstärkung um {res:.4g} dB")
                elif res < 0: print(f"  → Dämpfung um {abs(res):.4g} dB")
                else:         print(f"  → Kein Unterschied (x₁ = x₂)")

            elif x1 is not None and dB is not None:
                res = x1 * 10**(dB / factor)
                unit = 'V' if 'Spannung' in db_mode.value else ('A' if 'Strom' in db_mode.value else 'W')
                print(f"  x₂  = {fmt(res, unit)}")

            elif x2 is not None and dB is not None:
                res = x2 / 10**(dB / factor)
                unit = 'V' if 'Spannung' in db_mode.value else ('A' if 'Strom' in db_mode.value else 'W')
                print(f"  x₁  = {fmt(res, unit)}")

            else:
                print("ℹ️ Zwei Felder ausfüllen (x₁+x₂, x₁+dB, oder x₂+dB)")
        except Exception as e:
            print("❌ Fehler:", e)

btn_db.on_click(calc_db)
db_mode.observe(calc_db, names='value')
for fw in [db_x1, db_x2, db_dB]:
    fw.observe(calc_db, names='value')
display(widgets.VBox([db_mode, db_x1, db_x2, db_dB, btn_db, out_db]))

---
## 5.2 Wichtige dB-Werte auswendig lernen

Diese Werte sollte man im Schlaf kennen:

### Spannung / Strom (20·lg)
| dB | Verhältnis $U_2/U_1$ | Merkhilfe |
|----|----------------------|-----------|
| +20 dB | 10 | Zehnfache Spannung |
| +6 dB | $\approx 2$ | Doppelte Spannung |
| +3 dB | $\approx 1.414 = \sqrt{2}$ | Faktor $\sqrt{2}$ |
| 0 dB | 1 | Keine Änderung |
| −3 dB | $\approx 0.707 = 1/\sqrt{2}$ | Grenzfrequenz! |
| −6 dB | $\approx 0.5$ | Halbe Spannung |
| −20 dB | 0.1 | Zehntel Spannung |
| −40 dB | 0.01 | Hundertstel Spannung |

### Leistung (10·lg)
| dB | Verhältnis $P_2/P_1$ | Merkhilfe |
|----|----------------------|-----------|
| +10 dB | 10 | Zehnfache Leistung |
| +3 dB | $\approx 2$ | Doppelte Leistung |
| 0 dB | 1 | Keine Änderung |
| −3 dB | $\approx 0.5$ | Halbe Leistung |
| −10 dB | 0.1 | Zehntel Leistung |

> Der **−3 dB Punkt** ist die **Grenzfrequenz** bei Filtern!
> Dort ist $U = U_0 / \sqrt{2}$ und $P = P_0 / 2$.

In [ ]:
# ============================================================
# RECHNER 5.2: dB Merktabelle – Schnellreferenz
# ============================================================
display(HTML("<h3>🔢 Rechner 5.2 – Merktabellen Spannung & Leistung</h3>"))

out_tab = widgets.Output()
with out_tab:
    print("Spannung / Strom (20·lg):")
    print(f"  {'dB':>7}  {'Verhältnis':>12}  {'Kehrwert':>12}")
    print("  " + "-"*36)
    for db in [40, 20, 14, 6, 3, 0, -3, -6, -14, -20, -40]:
        ratio = 10**(db/20)
        print(f"  {db:>+7.0f} dB  {ratio:>12.5g}  {1/ratio:>12.5g}")
    print()
    print("Leistung (10·lg):")
    print(f"  {'dB':>7}  {'Verhältnis':>12}  {'Kehrwert':>12}")
    print("  " + "-"*36)
    for db in [20, 10, 3, 0, -3, -10, -20]:
        ratio = 10**(db/10)
        print(f"  {db:>+7.0f} dB  {ratio:>12.5g}  {1/ratio:>12.5g}")
display(out_tab)

---
## 5.3 Kettenrechnung – dB addieren

Der grosse Vorteil von dB: **Verstärkungen und Dämpfungen werden addiert statt multipliziert.**

$$L_{ges} = L_1 + L_2 + L_3 + \ldots \quad [dB]$$

**Beispiel:** Verstärker +26 dB → Kabel −3 dB → Filter −6 dB
$$L_{ges} = +26 - 3 - 6 = +17 \text{ dB}$$

### Absoluter Bezugspegel
| Einheit | Bezug | Verwendung |
|---------|-------|------------|
| dBm | 1 mW an 50 Ω | HF-Technik |
| dBV | 1 V | Audiotechnik |
| dBµV | 1 µV | Antennentechnik |

In [ ]:
# ============================================================
# RECHNER 5.3: Kettendämpfung – bis 6 Stufen
# ============================================================
display(HTML("<h3>🔢 Rechner 5.3 – Kettendämpfung / Gesamtverstärkung</h3>"
             "<i>dB-Werte eingeben (positiv = Verstärkung, negativ = Dämpfung)</i>"))

kette_fields = [make_input(f'Stufe {i+1} [dB] =', 'z.B. +20, -3, -6') for i in range(6)]
kette_U_ein  = make_input('U_ein [V]    =', 'optional → U_aus berechnen')
btn_kette    = make_button()
out_kette    = widgets.Output()

def calc_kette(_=None):
    out_kette.clear_output()
    with out_kette:
        try:
            dbs   = [p(f.value) for f in kette_fields]
            dbs_v = [d for d in dbs if d is not None]
            U_ein = p(kette_U_ein.value)
            if len(dbs_v) < 1:
                print("ℹ️ Mindestens eine Stufe eingeben"); return

            total = sum(dbs_v)
            print(f"  Stufen:   {' + '.join(f'{d:+.4g}' for d in dbs_v)} dB")
            print(f"  Gesamt:   {total:+.4g} dB")
            ratio_U = 10**(total/20)
            ratio_P = 10**(total/10)
            print(f"  U-Faktor: {ratio_U:.5g}  (20·lg)")
            print(f"  P-Faktor: {ratio_P:.5g}  (10·lg)")
            if total > 0:   print(f"  → Gesamtverstärkung")
            elif total < 0: print(f"  → Gesamtdämpfung")
            else:           print(f"  → Keine Gesamtänderung")
            if U_ein:
                U_aus = U_ein * ratio_U
                print(f"\n  U_ein = {fmt(U_ein, 'V')}")
                print(f"  U_aus = {fmt(U_aus, 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_kette.on_click(calc_kette)
for fw in kette_fields + [kette_U_ein]:
    fw.observe(calc_kette, names='value')
display(widgets.VBox(kette_fields + [kette_U_ein, btn_kette, out_kette]))

---
# Kapitel 6: Filter

Ein Filter lässt bestimmte Frequenzen durch und dämpft andere.
Die Grenzfrequenz $f_g$ liegt dort wo die Spannung auf $1/\sqrt{2} \approx 0.707$ gefallen ist (−3 dB Punkt).

---
## 6.1 Passiver RC-Tiefpass (1. Ordnung)

Ein einfacher Tiefpass lässt tiefe Frequenzen durch und dämpft hohe.

$$f_g = \frac{1}{2\pi \cdot R \cdot C}$$

**Übertragungsfunktion (Betrag):**
$$\left|\underline{H}\right| = \frac{1}{\sqrt{1 + \left(\frac{f}{f_g}\right)^2}}$$

**Phasengang:**
$$\varphi = -\arctan\!\left(\frac{f}{f_g}\right)$$

**Flankensteilheit:** −20 dB/Dekade (1. Ordnung)

<!-- Bild: RC Tiefpass Schaltung -->
<!-- Dateiname: rc_tiefpass_schaltung.png  → Google: "RC lowpass filter circuit diagram" -->

In [ ]:
# ============================================================
# RECHNER 6.1: Passiver RC-Tiefpass
# ============================================================
display(HTML("<h3>🔢 Rechner 6.1 – Passiver RC-Tiefpass</h3>"
             "<i>R und C eingeben – f_g und Verhalten bei Frequenz f werden berechnet</i>"))

tp_R  = make_input('R [Ω]   =')
tp_C  = make_input('C [F]   =', 'z.B. 100n')
tp_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
tp_U0 = make_input('U_ein [V]=', 'optional')
btn_tp = make_button()
out_tp = widgets.Output()

def calc_tp(_=None):
    out_tp.clear_output()
    with out_tp:
        try:
            R = p(tp_R.value); C = p(tp_C.value)
            f = p(tp_f.value); U0 = p(tp_U0.value)
            if any(x is None for x in [R, C]):
                print("ℹ️ R und C eingeben"); return
            fg = 1 / (2 * math.pi * R * C)
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            print(f"  τ     = R·C = {fmt(R*C, 's')}")
            if f:
                H     = 1 / math.sqrt(1 + (f/fg)**2)
                phi   = -math.degrees(math.atan(f/fg))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = {phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
                if f < fg:   print(f"  → Durchlassbereich (f < f_g)")
                elif f == fg: print(f"  → Genau bei Grenzfrequenz (−3 dB)")
                else:         print(f"  → Sperrbereich (f > f_g)")
        except Exception as e:
            print("❌ Fehler:", e)

btn_tp.on_click(calc_tp)
for fw in [tp_R, tp_C, tp_f, tp_U0]:
    fw.observe(calc_tp, names='value')
display(widgets.VBox([tp_R, tp_C, tp_f, tp_U0, btn_tp, out_tp]))

---
## 6.2 Passiver RC-Hochpass (1. Ordnung)

Ein Hochpass lässt hohe Frequenzen durch und dämpft tiefe.

$$f_g = \frac{1}{2\pi \cdot R \cdot C}$$

**Übertragungsfunktion:**
$$\left|\underline{H}\right| = \frac{f/f_g}{\sqrt{1 + \left(\frac{f}{f_g}\right)^2}}$$

**Phasengang:**
$$\varphi = +\arctan\!\left(\frac{f_g}{f}\right)$$

**Flankensteilheit:** +20 dB/Dekade unterhalb $f_g$

<!-- Bild: RC Hochpass Schaltung -->
<!-- Dateiname: rc_hochpass_schaltung.png  → Google: "RC highpass filter circuit diagram" -->

In [ ]:
# ============================================================
# RECHNER 6.2: Passiver RC-Hochpass
# ============================================================
display(HTML("<h3>🔢 Rechner 6.2 – Passiver RC-Hochpass</h3>"
             "<i>R und C eingeben – f_g und Verhalten bei Frequenz f werden berechnet</i>"))

hp_R  = make_input('R [Ω]   =')
hp_C  = make_input('C [F]   =', 'z.B. 100n')
hp_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
hp_U0 = make_input('U_ein [V]=', 'optional')
btn_hp = make_button()
out_hp = widgets.Output()

def calc_hp(_=None):
    out_hp.clear_output()
    with out_hp:
        try:
            R = p(hp_R.value); C = p(hp_C.value)
            f = p(hp_f.value); U0 = p(hp_U0.value)
            if any(x is None for x in [R, C]):
                print("ℹ️ R und C eingeben"); return
            fg = 1 / (2 * math.pi * R * C)
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            print(f"  τ     = R·C = {fmt(R*C, 's')}")
            if f:
                ratio = f / fg
                H     = ratio / math.sqrt(1 + ratio**2)
                phi   = math.degrees(math.atan(fg / f))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = +{phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
                if f > fg:   print(f"  → Durchlassbereich (f > f_g)")
                elif f == fg: print(f"  → Genau bei Grenzfrequenz (−3 dB)")
                else:         print(f"  → Sperrbereich (f < f_g)")
        except Exception as e:
            print("❌ Fehler:", e)

btn_hp.on_click(calc_hp)
for fw in [hp_R, hp_C, hp_f, hp_U0]:
    fw.observe(calc_hp, names='value')
display(widgets.VBox([hp_R, hp_C, hp_f, hp_U0, btn_hp, out_hp]))

---
## 6.3 Passiver RL-Tiefpass und Hochpass

Gleiche Logik wie RC – aber mit Spule statt Kondensator.

| | RL-Tiefpass | RL-Hochpass |
|---|---|---|
| Ausgang an | R | L |
| Durchlass | Tiefe f | Hohe f |
| Grenzfrequenz | $f_g = R / (2\pi L)$ | $f_g = R / (2\pi L)$ |

$$f_g = \frac{R}{2\pi \cdot L}$$

<!-- Bild: RL Tiefpass und Hochpass -->
<!-- Dateiname: rl_filter_schaltung.png  → Google: "RL lowpass highpass filter circuit diagram" -->

In [ ]:
# ============================================================
# RECHNER 6.3: Passiver RL-Filter (Tief- und Hochpass)
# ============================================================
display(HTML("<h3>🔢 Rechner 6.3 – Passiver RL-Filter</h3>"
             "<i>R und L eingeben</i>"))

rl_filt_mode = widgets.ToggleButtons(
    options=['Tiefpass (Ausgang an R)', 'Hochpass (Ausgang an L)'],
    description='Typ:', style={'description_width': '130px', 'button_width': '200px'}
)
rlf_R  = make_input('R [Ω]   =')
rlf_L  = make_input('L [H]   =', 'z.B. 10m')
rlf_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
rlf_U0 = make_input('U_ein [V]=', 'optional')
btn_rlf = make_button()
out_rlf = widgets.Output()

def calc_rlf(_=None):
    out_rlf.clear_output()
    with out_rlf:
        try:
            R = p(rlf_R.value); L = p(rlf_L.value)
            f = p(rlf_f.value); U0 = p(rlf_U0.value)
            if any(x is None for x in [R, L]):
                print("ℹ️ R und L eingeben"); return
            fg = R / (2 * math.pi * L)
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            if f:
                ratio = f / fg
                if 'Tiefpass' in rl_filt_mode.value:
                    H   = 1 / math.sqrt(1 + ratio**2)
                    phi = -math.degrees(math.atan(ratio))
                else:
                    H   = ratio / math.sqrt(1 + ratio**2)
                    phi = math.degrees(math.atan(fg / f))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = {phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_rlf.on_click(calc_rlf)
rl_filt_mode.observe(calc_rlf, names='value')
for fw in [rlf_R, rlf_L, rlf_f, rlf_U0]:
    fw.observe(calc_rlf, names='value')
display(widgets.VBox([rl_filt_mode, rlf_R, rlf_L, rlf_f, rlf_U0, btn_rlf, out_rlf]))

---
## 6.4 Aktiver Tiefpass mit OPV (Sallen-Key, 1. Ordnung)

Der OPV puffert den Ausgang – die Last beeinflusst den Filter nicht mehr.
Mit Verstärkung $A_0 = 1 + R_2/R_1$.

$$f_g = \frac{1}{2\pi \cdot R \cdot C}$$

$$A_0 = 1 + \frac{R_f}{R_1}$$

**Gesamtübertragung:**
$$\left|\underline{H}\right| = \frac{A_0}{\sqrt{1 + \left(\frac{f}{f_g}\right)^2}}$$

<!-- Bild: OPV Tiefpass 1. Ordnung nicht-invertierend -->
<!-- Dateiname: opv_tiefpass_1ordnung.png  → Google: "active low pass filter op-amp non-inverting first order circuit" -->

In [ ]:
# ============================================================
# RECHNER 6.4: Aktiver OPV-Tiefpass 1. Ordnung
# ============================================================
display(HTML("<h3>🔢 Rechner 6.4 – Aktiver OPV-Tiefpass (1. Ordnung)</h3>"
             "<i>R, C und Verstärkungsbeschaltung eingeben</i>"))

atp_R  = make_input('R [Ω]   =', 'Filterwiederstand')
atp_C  = make_input('C [F]   =', 'z.B. 100n')
atp_Rf = make_input('R_f [Ω] =', 'Gegenkopplungswid.')
atp_R1 = make_input('R₁ [Ω]  =', 'leer → A₀ = 1 (Spannungsfolger)')
atp_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
atp_U0 = make_input('U_ein [V]=', 'optional')
btn_atp = make_button()
out_atp = widgets.Output()

def calc_atp(_=None):
    out_atp.clear_output()
    with out_atp:
        try:
            R  = p(atp_R.value);  C  = p(atp_C.value)
            Rf = p(atp_Rf.value); R1 = p(atp_R1.value)
            f  = p(atp_f.value);  U0 = p(atp_U0.value)
            if any(x is None for x in [R, C]):
                print("ℹ️ R und C eingeben"); return
            fg = 1 / (2 * math.pi * R * C)
            if Rf and R1:   A0 = 1 + Rf / R1
            elif not Rf and not R1: A0 = 1.0
            else:
                print("ℹ️ Entweder beide (R_f und R₁) oder keinen eingeben"); return
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            print(f"  A₀    = {A0:.5g}  ({fmt(A0, '')} = {20*math.log10(A0):.3g} dB)")
            if f:
                H     = A0 / math.sqrt(1 + (f/fg)**2)
                phi   = -math.degrees(math.atan(f/fg))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = {phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_atp.on_click(calc_atp)
for fw in [atp_R, atp_C, atp_Rf, atp_R1, atp_f, atp_U0]:
    fw.observe(calc_atp, names='value')
display(widgets.VBox([atp_R, atp_C, atp_Rf, atp_R1, atp_f, atp_U0, btn_atp, out_atp]))

---
## 6.5 Aktiver Hochpass mit OPV (1. Ordnung)

Gleiche Struktur wie aktiver Tiefpass, R und C getauscht.

$$f_g = \frac{1}{2\pi \cdot R \cdot C} \qquad A_0 = 1 + \frac{R_f}{R_1}$$

$$\left|\underline{H}\right| = \frac{A_0 \cdot (f/f_g)}{\sqrt{1 + \left(\frac{f}{f_g}\right)^2}}$$

<!-- Bild: OPV Hochpass 1. Ordnung -->
<!-- Dateiname: opv_hochpass_1ordnung.png  → Google: "active high pass filter op-amp first order circuit diagram" -->

In [ ]:
# ============================================================
# RECHNER 6.5: Aktiver OPV-Hochpass 1. Ordnung
# ============================================================
display(HTML("<h3>🔢 Rechner 6.5 – Aktiver OPV-Hochpass (1. Ordnung)</h3>"
             "<i>R, C und Verstärkungsbeschaltung eingeben</i>"))

ahp_R  = make_input('R [Ω]   =')
ahp_C  = make_input('C [F]   =', 'z.B. 100n')
ahp_Rf = make_input('R_f [Ω] =', 'Gegenkopplungswid.')
ahp_R1 = make_input('R₁ [Ω]  =', 'leer → A₀ = 1')
ahp_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
ahp_U0 = make_input('U_ein [V]=', 'optional')
btn_ahp = make_button()
out_ahp = widgets.Output()

def calc_ahp(_=None):
    out_ahp.clear_output()
    with out_ahp:
        try:
            R  = p(ahp_R.value);  C  = p(ahp_C.value)
            Rf = p(ahp_Rf.value); R1 = p(ahp_R1.value)
            f  = p(ahp_f.value);  U0 = p(ahp_U0.value)
            if any(x is None for x in [R, C]):
                print("ℹ️ R und C eingeben"); return
            fg = 1 / (2 * math.pi * R * C)
            if Rf and R1:           A0 = 1 + Rf / R1
            elif not Rf and not R1: A0 = 1.0
            else:
                print("ℹ️ Entweder beide (R_f und R₁) oder keinen eingeben"); return
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            print(f"  A₀    = {A0:.5g}  ({20*math.log10(A0):.3g} dB)")
            if f:
                ratio  = f / fg
                H      = A0 * ratio / math.sqrt(1 + ratio**2)
                phi    = math.degrees(math.atan(fg / f))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = +{phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_ahp.on_click(calc_ahp)
for fw in [ahp_R, ahp_C, ahp_Rf, ahp_R1, ahp_f, ahp_U0]:
    fw.observe(calc_ahp, names='value')
display(widgets.VBox([ahp_R, ahp_C, ahp_Rf, ahp_R1, ahp_f, ahp_U0, btn_ahp, out_ahp]))

---
## 6.6 Aktiver Bandpass mit OPV (2. Ordnung)

Lässt einen Frequenzbereich um $f_0$ durch, dämpft darunter und darüber.

$$f_0 = \frac{1}{2\pi \cdot \sqrt{R_1 \cdot R_2 \cdot C_1 \cdot C_2}}$$

Vereinfacht mit $R_1 = R_2 = R$ und $C_1 = C_2 = C$:
$$f_0 = \frac{1}{2\pi \cdot R \cdot C}$$

**Güte und Bandbreite:**
$$Q = \frac{f_0}{\Delta f} \qquad \Delta f = \frac{f_0}{Q}$$

**Mittenverstärkung** (bei $f_0$):
$$A_0 = \frac{R_2}{2 \cdot R_1}$$

**Flankensteilheit:** +20 dB/Dek. unterhalb, −20 dB/Dek. oberhalb $f_0$

<!-- Bild: OPV Bandpass MFB (Multiple Feedback) -->
<!-- Dateiname: opv_bandpass_mfb.png  → Google: "active bandpass filter op-amp multiple feedback MFB circuit" -->

In [ ]:
# ============================================================
# RECHNER 6.6: Aktiver OPV-Bandpass (vereinfacht R1=R2, C1=C2)
# ============================================================
display(HTML("<h3>🔢 Rechner 6.6 – Aktiver OPV-Bandpass</h3>"
             "<i>R, C und Q eingeben – oder f₀ direkt</i>"))

bp_R  = make_input('R [Ω]   =', 'R₁ = R₂')
bp_C  = make_input('C [F]   =', 'C₁ = C₂, z.B. 100n')
bp_Q  = make_input('Q       =', 'Güte, z.B. 5')
bp_f  = make_input('f [Hz]  =', 'optional: Verhalten bei f')
bp_U0 = make_input('U_ein [V]=', 'optional')
btn_bp = make_button()
out_bp = widgets.Output()

def calc_bp(_=None):
    out_bp.clear_output()
    with out_bp:
        try:
            R  = p(bp_R.value); C  = p(bp_C.value)
            Q  = p(bp_Q.value); f  = p(bp_f.value)
            U0 = p(bp_U0.value)
            if any(x is None for x in [R, C]):
                print("ℹ️ R und C eingeben"); return
            f0 = 1 / (2 * math.pi * R * C)
            Qv = Q if Q else 1/math.sqrt(2)
            df = f0 / Qv
            fu = f0 - df/2
            fo = f0 + df/2
            print(f"  f₀    = {fmt(f0, 'Hz')}")
            print(f"  Q     = {Qv:.4g}")
            print(f"  Δf    = {fmt(df, 'Hz')}")
            print(f"  f_u   = {fmt(fu, 'Hz')}")
            print(f"  f_o   = {fmt(fo, 'Hz')}")
            if f:
                # Bandpass Übertragungsfunktion 2. Ordnung
                norm  = f / f0
                H     = (1/Qv * norm) / math.sqrt((1 - norm**2)**2 + (norm/Qv)**2)
                dB_val = 20 * math.log10(H) if H > 0 else -99
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
                if abs(f - f0) < df/2:
                    print(f"  → Im Durchlassbereich")
                else:
                    print(f"  → Im Sperrbereich")
        except Exception as e:
            print("❌ Fehler:", e)

btn_bp.on_click(calc_bp)
for fw in [bp_R, bp_C, bp_Q, bp_f, bp_U0]:
    fw.observe(calc_bp, names='value')
display(widgets.VBox([bp_R, bp_C, bp_Q, bp_f, bp_U0, btn_bp, out_bp]))

---
## 6.7 Sallen-Key Filter (2. Ordnung)

Der **Sallen-Key** ist die häufigste Topologie für aktive Filter 2. Ordnung.
Er realisiert Tief- oder Hochpass mit wählbarer Charakteristik.

### Tiefpass Sallen-Key (R₁=R₂=R, C₁=C₂=C)
$$f_g = \frac{1}{2\pi \cdot R \cdot C}$$

$$Q = \frac{1}{3 - A_0} \qquad A_0 = 1 + \frac{R_f}{R_1}$$

**Wichtige Charakteristiken:**

| Charakteristik | Q | A₀ | Eigenschaft |
|---|---|---|---|
| Butterworth | 0.707 | 1.586 | Maximale Flachheit im Durchlass |
| Bessel | 0.577 | 1.268 | Bestes Phasenverhalten |
| Chebyshev 1 dB | 0.956 | 2.234 | Steilste Flanke |

**Flankensteilheit:** −40 dB/Dekade (2. Ordnung = doppelt so steil wie 1. Ordnung!)

<!-- Bild: Sallen-Key Tiefpass Schaltung -->
<!-- Dateiname: sallen_key_tiefpass.png  → Google: "Sallen-Key low pass filter circuit second order op-amp" -->

In [ ]:
# ============================================================
# RECHNER 6.7: Sallen-Key Tiefpass 2. Ordnung
# ============================================================
display(HTML("<h3>🔢 Rechner 6.7 – Sallen-Key Tiefpass (2. Ordnung)</h3>"
             "<i>R, C und Charakteristik wählen</i>"))

sk_char = widgets.Dropdown(
    options=[('Butterworth  (Q=0.707, A₀=1.586)', (0.7071, 1.5858)),
             ('Bessel       (Q=0.577, A₀=1.268)', (0.5774, 1.2679)),
             ('Chebyshev 1dB(Q=0.956, A₀=2.234)', (0.9565, 2.2346)),
             ('Manuell eingeben',                  None)],
    description='Charakteristik:', style={'description_width': '160px'},
    layout=widgets.Layout(width='380px')
)
sk_R   = make_input('R [Ω]   =', 'R₁ = R₂')
sk_C   = make_input('C [F]   =', 'C₁ = C₂, z.B. 100n')
sk_Q   = make_input('Q       =', 'nur bei Manuell')
sk_f   = make_input('f [Hz]  =', 'optional: Verhalten bei f')
sk_U0  = make_input('U_ein [V]=', 'optional')
btn_sk  = make_button()
out_sk  = widgets.Output()

def calc_sk(_=None):
    out_sk.clear_output()
    with out_sk:
        try:
            R  = p(sk_R.value); C = p(sk_C.value)
            f  = p(sk_f.value); U0 = p(sk_U0.value)
            char_val = sk_char.value
            if any(x is None for x in [R, C]):
                print("ℹ️ R und C eingeben"); return
            if char_val is not None:
                Qv, A0 = char_val
            else:
                Qv = p(sk_Q.value)
                if Qv is None:
                    print("ℹ️ Q eingeben oder Charakteristik wählen"); return
                A0 = 3 - 1/Qv
            fg = 1 / (2 * math.pi * R * C)
            Rf_R1 = A0 - 1
            print(f"  f_g   = {fmt(fg, 'Hz')}")
            print(f"  Q     = {Qv:.4g}")
            print(f"  A₀    = {A0:.4g}  (R_f/R₁ = {Rf_R1:.4g})")
            if A0 >= 3:
                print(f"  ⚠️ A₀ ≥ 3 → instabil! Q nicht realisierbar.")
                return
            if f:
                norm   = f / fg
                denom  = math.sqrt((1 - norm**2)**2 + (norm/Qv)**2)
                H      = A0 / denom
                phi    = -math.degrees(math.atan2(norm/Qv, 1 - norm**2))
                dB_val = 20 * math.log10(H)
                print(f"\n  Bei f = {fmt(f, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                print(f"  φ     = {phi:.3g}°")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_sk.on_click(calc_sk)
sk_char.observe(calc_sk, names='value')
for fw in [sk_R, sk_C, sk_Q, sk_f, sk_U0]:
    fw.observe(calc_sk, names='value')
display(widgets.VBox([sk_char, sk_R, sk_C, sk_Q, sk_f, sk_U0, btn_sk, out_sk]))

---
## 6.8 Notch-Filter (Kerbfilter) mit OPV

Ein **Notch-Filter** (Bandsperre) dämpft eine einzelne Frequenz sehr stark
und lässt alle anderen durch. Typische Anwendung: 50 Hz Netzstörungen unterdrücken.

**Twin-T Notch:**
$$f_0 = \frac{1}{2\pi \cdot R \cdot C}$$

Mit OPV-Gegenkopplung ist die Kerbtiefe einstellbar.
Ohne Gegenkopplung (passiv): Kerbtiefe theoretisch $-\infty$ dB bei exakten Bauteilen.

$$\left|\underline{H}\right| = \frac{\left|1 - \left(\frac{f}{f_0}\right)^2\right|}{\sqrt{\left(1-\left(\frac{f}{f_0}\right)^2\right)^2 + \left(\frac{f}{f_0 \cdot Q}\right)^2}}$$

<!-- Bild: Twin-T Notch Filter mit OPV -->
<!-- Dateiname: twin_t_notch_filter_opv.png  → Google: "Twin-T notch filter op-amp active circuit 50Hz" -->

In [ ]:
# ============================================================
# RECHNER 6.8: Notch-Filter (Twin-T)
# ============================================================
display(HTML("<h3>🔢 Rechner 6.8 – Notch-Filter (Twin-T)</h3>"
             "<i>R, C und Q eingeben</i>"))

nt_R  = make_input('R [Ω]   =')
nt_C  = make_input('C [F]   =', 'z.B. 100n')
nt_Q  = make_input('Q       =', 'leer → passiv (Q≈0.25)')
nt_f  = make_input('f [Hz]  =', 'optional: Dämpfung bei f')
nt_U0 = make_input('U_ein [V]=', 'optional')
btn_nt = make_button()
out_nt = widgets.Output()

def calc_nt(_=None):
    out_nt.clear_output()
    with out_nt:
        try:
            R  = p(nt_R.value); C  = p(nt_C.value)
            Q  = p(nt_Q.value); fv = p(nt_f.value)
            U0 = p(nt_U0.value)
            if any(x is None for x in [R, C]):
                print("ℹ️ R und C eingeben"); return
            f0 = 1 / (2 * math.pi * R * C)
            Qv = Q if Q else 0.25
            df = f0 / Qv
            print(f"  f₀ (Kerbe) = {fmt(f0, 'Hz')}")
            print(f"  Q          = {Qv:.4g}")
            print(f"  Δf (−3dB)  = {fmt(df, 'Hz')}")
            if not Q:
                print(f"  (passiver Twin-T, Q ≈ 0.25)")
            if fv:
                norm   = fv / f0
                num    = abs(1 - norm**2)
                denom  = math.sqrt((1 - norm**2)**2 + (norm/Qv)**2)
                H      = num / denom if denom > 0 else 0
                dB_val = 20 * math.log10(H) if H > 1e-10 else -100
                print(f"\n  Bei f = {fmt(fv, 'Hz')}:")
                print(f"  |H|   = {H:.5g}  ({dB_val:.3g} dB)")
                if U0:
                    print(f"  U_aus = {fmt(H * U0, 'V')}")
                if abs(fv - f0) < 1:
                    print(f"  → Genau bei Kerbfrequenz – maximale Dämpfung")
        except Exception as e:
            print("❌ Fehler:", e)

btn_nt.on_click(calc_nt)
for fw in [nt_R, nt_C, nt_Q, nt_f, nt_U0]:
    fw.observe(calc_nt, names='value')
display(widgets.VBox([nt_R, nt_C, nt_Q, nt_f, nt_U0, btn_nt, out_nt]))

---
# Kapitel 7: Transformator

---
## 7.1 Grundlagen und Übersetzungsverhältnis

Ein **Transformator** überträgt elektrische Energie durch magnetische Kopplung
zwischen zwei Wicklungen – galvanisch getrennt und verlustarm.

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Primärspannung | $U_1$ | V |
| Sekundärspannung | $U_2$ | V |
| Primärstrom | $I_1$ | A |
| Sekundärstrom | $I_2$ | A |
| Primärwindungen | $N_1$ | – |
| Sekundärwindungen | $N_2$ | – |
| Übersetzungsverhältnis | $ü$ | – |

### Übersetzungsverhältnis
$$ü = \frac{N_1}{N_2} = \frac{U_1}{U_2} = \frac{I_2}{I_1}$$

> Achtung: Beim Strom steht $I_2/I_1$ – also umgekehrt zu Spannung und Windungen!

<!-- Bild: Transformator Schaltzeichen mit N1, N2, U1, U2, I1, I2 -->
<!-- Dateiname: transformator_schaltzeichen.png  → Google: "transformer schematic symbol primary secondary winding N1 N2" -->

In [ ]:
# ============================================================
# RECHNER 7.1: Übersetzungsverhältnis
# ============================================================
display(HTML("<h3>🔢 Rechner 7.1 – Übersetzungsverhältnis</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

ue_N1 = make_input('N₁ (Primär)   =', 'Windungszahl')
ue_N2 = make_input('N₂ (Sekundär) =', 'Windungszahl')
ue_U1 = make_input('U₁ [V]        =')
ue_U2 = make_input('U₂ [V]        =')
ue_I1 = make_input('I₁ [A]        =')
ue_I2 = make_input('I₂ [A]        =')
btn_ue = make_button()
out_ue = widgets.Output()

def calc_ue(_=None):
    out_ue.clear_output()
    with out_ue:
        try:
            N1 = p(ue_N1.value); N2 = p(ue_N2.value)
            U1 = p(ue_U1.value); U2 = p(ue_U2.value)
            I1 = p(ue_I1.value); I2 = p(ue_I2.value)

            # ü aus vorhandenen Paaren bestimmen
            ue = None
            if N1 and N2: ue = N1 / N2; print(f"  ü = N₁/N₂ = {ue:.5g}")
            if U1 and U2:
                ue_u = U1 / U2
                print(f"  ü = U₁/U₂ = {ue_u:.5g}")
                if ue and abs(ue - ue_u) > 0.001:
                    print(f"  ⚠️ Widerspruch: N-Verhältnis ≠ U-Verhältnis!")
                ue = ue_u
            if I1 and I2:
                ue_i = I2 / I1
                print(f"  ü = I₂/I₁ = {ue_i:.5g}")
                if ue and abs(ue - ue_i) > 0.001:
                    print(f"  ⚠️ Widerspruch: Verhältnis passt nicht!")
                ue = ue_i

            if ue is None:
                print("ℹ️ Mindestens ein Paar (N₁+N₂, U₁+U₂ oder I₁+I₂) eingeben")
                return

            print(f"\n  ü     = {ue:.5g}")
            if   ue > 1: print(f"  → Abwärtstransformator (U₂ < U₁)")
            elif ue < 1: print(f"  → Aufwärtstransformator (U₂ > U₁)")
            else:        print(f"  → 1:1 Trenntransformator")

            # Fehlende Werte berechnen
            if ue:
                if U1 and not U2: print(f"  U₂ = U₁/ü = {fmt(U1/ue, 'V')}")
                if U2 and not U1: print(f"  U₁ = U₂·ü = {fmt(U2*ue, 'V')}")
                if I1 and not I2: print(f"  I₂ = I₁·ü = {fmt(I1*ue, 'A')}")
                if I2 and not I1: print(f"  I₁ = I₂/ü = {fmt(I2/ue, 'A')}")
                if N1 and not N2: print(f"  N₂ = N₁/ü = {N1/ue:.4g}")
                if N2 and not N1: print(f"  N₁ = N₂·ü = {N2*ue:.4g}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_ue.on_click(calc_ue)
for fw in [ue_N1, ue_N2, ue_U1, ue_U2, ue_I1, ue_I2]:
    fw.observe(calc_ue, names='value')
display(widgets.VBox([ue_N1, ue_N2, ue_U1, ue_U2, ue_I1, ue_I2, btn_ue, out_ue]))

---
## 7.2 Leistung und Wirkungsgrad

Beim idealen Transformator gilt:
$$P_1 = P_2 \qquad U_1 \cdot I_1 = U_2 \cdot I_2$$

Beim realen Transformator entstehen Verluste:

| Verlusttyp | Ursache | Wo |
|------------|---------|-----|
| Kupferverluste $P_{Cu}$ | Wicklungswiderstand $R_{Cu}$ | Wicklung |
| Eisenverluste $P_{Fe}$ | Ummagnetisierung, Wirbelströme | Kern |

### Wirkungsgrad
$$\eta = \frac{P_2}{P_1} = \frac{P_2}{P_2 + P_{Cu} + P_{Fe}}$$

Gute Netztransformatoren erreichen $\eta = 95\ldots99\,\%$.

In [ ]:
# ============================================================
# RECHNER 7.2: Leistung und Wirkungsgrad
# ============================================================
display(HTML("<h3>🔢 Rechner 7.2 – Leistung und Wirkungsgrad</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

eta_P1   = make_input('P₁ [W]    =', 'Primärleistung')
eta_P2   = make_input('P₂ [W]    =', 'Sekundärleistung')
eta_Pcu  = make_input('P_Cu [W]  =', 'Kupferverluste (optional)')
eta_Pfe  = make_input('P_Fe [W]  =', 'Eisenverluste (optional)')
eta_eta  = make_input('η [%]     =', 'Wirkungsgrad z.B. 97')
btn_eta  = make_button()
out_eta  = widgets.Output()

def calc_eta(_=None):
    out_eta.clear_output()
    with out_eta:
        try:
            P1  = p(eta_P1.value)
            P2  = p(eta_P2.value)
            Pcu = p(eta_Pcu.value)
            Pfe = p(eta_Pfe.value)
            eta_val = p(eta_eta.value)
            if eta_val: eta_val = eta_val / 100  # % → Faktor

            vals = {'P₁': P1, 'P₂': P2, 'η': eta_val}
            miss = [k for k, v in vals.items() if v is None]

            Pv = (Pcu if Pcu else 0) + (Pfe if Pfe else 0)

            if len(miss) == 0:
                print("❌ Ein Feld leer lassen!")
                return
            if len(miss) > 1:
                print("ℹ️ Mindestens zwei Felder (P₁, P₂, η) ausfüllen")
                return

            m = miss[0]
            if   m == 'η':  res = P2 / P1;       print(f"  η  = P₂/P₁ = {res*100:.4g} %")
            elif m == 'P₁': res = P2 / eta_val;  print(f"  P₁ = P₂/η  = {fmt(res, 'W')}")
            elif m == 'P₂': res = P1 * eta_val;  print(f"  P₂ = P₁·η  = {fmt(res, 'W')}")

            P1v = P1 if P1 else res if m == 'P₁' else None
            P2v = P2 if P2 else res if m == 'P₂' else None
            ev  = eta_val if eta_val else res if m == 'η' else None

            if P1v and P2v:
                Pverl = P1v - P2v
                print(f"\n  P_Verl = P₁ - P₂ = {fmt(Pverl, 'W')}")
                if Pcu: print(f"  P_Cu   = {fmt(Pcu, 'W')}")
                if Pfe: print(f"  P_Fe   = {fmt(Pfe, 'W')}")
                if Pcu or Pfe:
                    print(f"  Σ Verl = {fmt(Pv, 'W')}  (eingegeben)")
            if ev:
                print(f"\n  η      = {ev*100:.4g} %")
                if   ev >= 0.98: print(f"  → Sehr guter Transformator")
                elif ev >= 0.94: print(f"  → Guter Transformator")
                else:            print(f"  → ⚠️ Hohe Verluste")
        except Exception as e:
            print("❌ Fehler:", e)

btn_eta.on_click(calc_eta)
for fw in [eta_P1, eta_P2, eta_Pcu, eta_Pfe, eta_eta]:
    fw.observe(calc_eta, names='value')
display(widgets.VBox([eta_P1, eta_P2, eta_Pcu, eta_Pfe, eta_eta, btn_eta, out_eta]))

---
## 7.3 Widerstandstransformation

Ein Transformator transformiert nicht nur Spannung und Strom,
sondern auch **Widerstände** – sehr wichtig für Impedanzanpassung!

$$R_1' = ü^2 \cdot R_2 \qquad \text{(Lastwiderstand auf Primärseite umgerechnet)}$$

$$R_2' = \frac{R_1}{ü^2} \qquad \text{(Primärwiderstand auf Sekundärseite)}$$

**Anwendung:** Lautsprecher-Impedanzanpassung, HF-Anpassung

<!-- Bild: Transformator Widerstandstransformation -->
<!-- Dateiname: transformator_impedanzanpassung.png  → Google: "transformer impedance matching circuit diagram R primary secondary" -->

In [ ]:
# ============================================================
# RECHNER 7.3: Widerstandstransformation
# ============================================================
display(HTML("<h3>🔢 Rechner 7.3 – Widerstandstransformation</h3>"
             "<i>ü und einen Widerstand eingeben</i>"))

wt_ue  = make_input('ü = N₁/N₂  =', 'Übersetzungsverhältnis')
wt_R1  = make_input('R₁ [Ω]     =', 'Primärwiderstand')
wt_R2  = make_input('R₂ [Ω]     =', 'Sekundärlast')
btn_wt = make_button()
out_wt = widgets.Output()

def calc_wt(_=None):
    out_wt.clear_output()
    with out_wt:
        try:
            ue = p(wt_ue.value)
            R1 = p(wt_R1.value)
            R2 = p(wt_R2.value)
            if ue is None:
                print("ℹ️ ü eingeben"); return
            if R2 is not None:
                R1_trans = ue**2 * R2
                print(f"  R₂ = {fmt(R2, 'Ω')}  auf Primärseite:")
                print(f"  R₁' = ü²·R₂ = {fmt(R1_trans, 'Ω')}")
            if R1 is not None:
                R2_trans = R1 / ue**2
                print(f"  R₁ = {fmt(R1, 'Ω')}  auf Sekundärseite:")
                print(f"  R₂' = R₁/ü² = {fmt(R2_trans, 'Ω')}")
            if R1 is None and R2 is None:
                print("ℹ️ R₁ oder R₂ eingeben")
        except Exception as e:
            print("❌ Fehler:", e)

btn_wt.on_click(calc_wt)
for fw in [wt_ue, wt_R1, wt_R2]:
    fw.observe(calc_wt, names='value')
display(widgets.VBox([wt_ue, wt_R1, wt_R2, btn_wt, out_wt]))

---
## 7.4 Leerlauf und Kurzschluss

### Leerlauf (Sekundär offen, $I_2 = 0$)
- Primärstrom = kleiner Magnetisierungsstrom $I_\mu$
- $U_2 = U_1 / ü$ (Leerlaufspannung)
- Keine Leistungsübertragung (nur Eisenverluste)

### Kurzschluss (Sekundär kurzgeschlossen, $U_2 = 0$)
$$I_{K2} = \frac{U_1}{ü \cdot (R_{Cu1} + ü^2 \cdot R_{Cu2})}$$

> ⚠️ Kurzschlussstrom kann sehr gross sein – Sicherung nötig!

### Spannungsabfall unter Last (Näherung)
$$\Delta U = I_2 \cdot R_{Cu2} \qquad \text{(Sekundärseitig)}$$

$$U_{2,Last} \approx U_{2,Leerlauf} - \Delta U$$

In [ ]:
# ============================================================
# RECHNER 7.4: Spannungsabfall unter Last
# ============================================================
display(HTML("<h3>🔢 Rechner 7.4 – Spannungsabfall unter Last</h3>"
             "<i>Sekundärspannung, Wicklungswiderstand und Last eingeben</i>"))

last_U2ll  = make_input('U₂ Leerlauf [V] =')
last_Rcu2  = make_input('R_Cu2 [Ω]       =', 'Sek. Wicklungswid.')
last_I2    = make_input('I₂ [A]          =', 'Laststrom')
last_Rlast = make_input('R_Last [Ω]      =', 'oder Lastwid. statt I₂')
btn_last   = make_button()
out_last   = widgets.Output()

def calc_last(_=None):
    out_last.clear_output()
    with out_last:
        try:
            U2ll  = p(last_U2ll.value)
            Rcu2  = p(last_Rcu2.value)
            I2    = p(last_I2.value)
            Rlast = p(last_Rlast.value)

            if U2ll is None or Rcu2 is None:
                print("ℹ️ U₂ Leerlauf und R_Cu2 eingeben"); return

            if I2 is None and Rlast is not None:
                I2 = U2ll / (Rcu2 + Rlast)
                print(f"  I₂    = U₂ll/(R_Cu2+R_Last) = {fmt(I2, 'A')}")
            elif I2 is None:
                print("ℹ️ I₂ oder R_Last eingeben"); return

            dU   = I2 * Rcu2
            U2l  = U2ll - dU
            P2   = U2l * I2
            Pcu  = I2**2 * Rcu2

            print(f"  ΔU    = I₂·R_Cu2 = {fmt(dU, 'V')}")
            print(f"  U₂ Last = {fmt(U2l, 'V')}")
            print(f"  I₂      = {fmt(I2, 'A')}")
            print(f"  P₂      = {fmt(P2, 'W')}")
            print(f"  P_Cu2   = {fmt(Pcu, 'W')}")
            abw = dU / U2ll * 100
            print(f"  Spannungsabfall: {abw:.3g} %")
            if abw > 10:
                print(f"  ⚠️ Hoher Spannungsabfall – R_Cu2 zu gross?")
        except Exception as e:
            print("❌ Fehler:", e)

btn_last.on_click(calc_last)
for fw in [last_U2ll, last_Rcu2, last_I2, last_Rlast]:
    fw.observe(calc_last, names='value')
display(widgets.VBox([last_U2ll, last_Rcu2, last_I2, last_Rlast, btn_last, out_last]))

---
## 7.5 Spartrafo und Trenntransformator

### Spartrafo
Primär- und Sekundärwicklung sind **nicht galvanisch getrennt** – eine gemeinsame Wicklung.

- Vorteil: Kleiner, leichter, günstiger
- Nachteil: Keine galvanische Trennung → Gefahr bei Berührung!
- Typisch: Stelltransformator (Variac)

$$ü = \frac{N_{ges}}{N_2} \qquad \text{oder} \qquad ü = \frac{N_1}{N_{ges}}$$

### Trenntransformator ($ü = 1$)
- $U_2 = U_1$, keine Spannungsänderung
- Zweck: Galvanische Trennung (Schutzklasse, Messtechnik)
- Schutz vor Netzpotential

<!-- Bild: Spartrafo vs Trenntransformator Vergleich -->
<!-- Dateiname: spartrafo_trenntransformator.png  → Google: "autotransformer vs isolation transformer schematic comparison" -->

---
# Kapitel 8: Gleichrichter und Siebung

---
## 8.1 Grundlagen der Gleichrichtung

Ein **Gleichrichter** wandelt Wechselspannung (AC) in pulsierende Gleichspannung (DC) um.
Die nachfolgende **Siebung** glättet die Restwelligkeit.

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Spitzenwert | $\hat{U}$ | V |
| Effektivwert | $U_{eff}$ | V |
| Gleichspannung (Mittelwert) | $\bar{U}$ | V |
| Brummspannung | $U_{Br}$ | V |
| Welligkeit | $w$ | % |

### Wichtige Zusammenhänge Sinusspannung
$$\hat{U} = U_{eff} \cdot \sqrt{2} \qquad U_{eff} = \frac{\hat{U}}{\sqrt{2}}$$

<!-- Bild: Sinusspannung mit Spitzen- und Effektivwert -->
<!-- Dateiname: sinusspannung_scheitel_effektiv.png  → Google: "AC sine wave peak RMS voltage diagram labeled" -->

In [ ]:
# ============================================================
# RECHNER 8.1: Spitzenwert / Effektivwert Umrechnung
# ============================================================
display(HTML("<h3>🔢 Rechner 8.1 – Spitzenwert / Effektivwert</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

eff_Up  = make_input('Û (Spitzenwert) [V] =')
eff_Ue  = make_input('U_eff [V]           =')
btn_eff = make_button()
out_eff = widgets.Output()

def calc_eff(_=None):
    out_eff.clear_output()
    with out_eff:
        try:
            Up = p(eff_Up.value); Ue = p(eff_Ue.value)
            vals = {'Û': Up, 'U_eff': Ue}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if m == 'Û':
                res = Ue * math.sqrt(2)
                print(f"  Û     = U_eff · √2 = {fmt(res, 'V')}")
            else:
                res = Up / math.sqrt(2)
                print(f"  U_eff = Û / √2     = {fmt(res, 'V')}")
            Upv = Up if Up else res if m == 'Û' else None
            Uev = Ue if Ue else res if m == 'U_eff' else None
            if Upv and Uev:
                print(f"\n  Zur Kontrolle:")
                print(f"  Û     = {fmt(Upv, 'V')}")
                print(f"  U_eff = {fmt(Uev, 'V')}")
                print(f"  Faktor √2 = {Upv/Uev:.6g}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_eff.on_click(calc_eff)
for fw in [eff_Up, eff_Ue]:
    fw.observe(calc_eff, names='value')
display(widgets.VBox([eff_Up, eff_Ue, btn_eff, out_eff]))

---
## 8.2 Einweggleichrichter (M1)

Nur die **positive Halbwelle** wird durchgelassen. Einfachste Schaltung, aber schlechte Ausnutzung.

$$\bar{U} = \frac{\hat{U}}{\pi} \approx 0.318 \cdot \hat{U}$$

$$\bar{U} = 0.45 \cdot U_{eff}$$

**Eigenschaften:**
- Nur 50% der Halbwellen genutzt
- Brummfrequenz = Netzfrequenz ($f_{Br} = f_{Netz}$)
- Grosse Restwelligkeit
- Diodenspannung $U_F \approx 0.7\,V$ subtrahieren!

$$\bar{U}_{real} = \bar{U} - U_F$$

<!-- Bild: Einweggleichrichter Schaltung und Ausgangsspannung -->
<!-- Dateiname: einweggleichrichter_schaltung.png  → Google: "half wave rectifier circuit diagram output waveform" -->

In [ ]:
# ============================================================
# RECHNER 8.2: Einweggleichrichter (M1)
# ============================================================
display(HTML("<h3>🔢 Rechner 8.2 – Einweggleichrichter (M1)</h3>"
             "<i>Eingangsspannung eingeben</i>"))

m1_Ueff = make_input('U_eff (AC) [V] =', 'z.B. 230 für Netz')
m1_Up   = make_input('Û (AC) [V]    =', 'oder Spitzenwert')
m1_UF   = make_input('U_F Diode [V] =', 'Flussspannung, ca. 0.7')
btn_m1  = make_button()
out_m1  = widgets.Output()

def calc_m1(_=None):
    out_m1.clear_output()
    with out_m1:
        try:
            Ueff = p(m1_Ueff.value)
            Up   = p(m1_Up.value)
            UF   = p(m1_UF.value) if m1_UF.value.strip() else 0.7

            if Ueff is None and Up is None:
                print("ℹ️ U_eff oder Û eingeben"); return
            if Up is None:   Up   = Ueff * math.sqrt(2)
            if Ueff is None: Ueff = Up / math.sqrt(2)

            U_avg      = Up / math.pi
            U_avg_real = U_avg - UF
            f_Br       = 50  # Hz bei 50 Hz Netz

            print(f"  Û          = {fmt(Up, 'V')}")
            print(f"  U_eff      = {fmt(Ueff, 'V')}")
            print(f"  Ū (ideal)  = Û/π = {fmt(U_avg, 'V')}")
            print(f"  U_F        = {fmt(UF, 'V')}")
            print(f"  Ū (real)   = {fmt(U_avg_real, 'V')}")
            print(f"  f_Brumm    = {f_Br} Hz  (= Netzfrequenz)")
            print(f"  Ausnutzung = 50 %  (nur positive HW)")
            if U_avg_real < 0:
                print(f"  ⚠️ Ausgangsspannung negativ – Eingangsspannung zu klein!")
        except Exception as e:
            print("❌ Fehler:", e)

btn_m1.on_click(calc_m1)
for fw in [m1_Ueff, m1_Up, m1_UF]:
    fw.observe(calc_m1, names='value')
display(widgets.VBox([m1_Ueff, m1_Up, m1_UF, btn_m1, out_m1]))

---
## 8.3 Brückengleichrichter / Graetz-Schaltung (B2)

**Beide** Halbwellen werden gleichgerichtet – deutlich besser als M1.

$$\bar{U} = \frac{2 \cdot \hat{U}}{\pi} \approx 0.637 \cdot \hat{U}$$

$$\bar{U} = 0.9 \cdot U_{eff}$$

**Eigenschaften:**
- Beide Halbwellen genutzt → doppelte Ausnutzung
- Brummfrequenz = **doppelte** Netzfrequenz ($f_{Br} = 2 \cdot f_{Netz} = 100\,Hz$)
- Immer **2 Dioden** in Reihe leitend → $2 \cdot U_F$ abziehen!

$$\bar{U}_{real} = \frac{2 \cdot \hat{U}}{\pi} - 2 \cdot U_F$$

<!-- Bild: Brückengleichrichter Graetz Schaltung -->
<!-- Dateiname: brueckengleichrichter_graetz.png  → Google: "bridge rectifier Graetz circuit diagram B2 full wave" -->

In [ ]:
# ============================================================
# RECHNER 8.3: Brückengleichrichter B2 (Graetz)
# ============================================================
display(HTML("<h3>🔢 Rechner 8.3 – Brückengleichrichter B2 (Graetz)</h3>"
             "<i>Eingangsspannung eingeben</i>"))

b2_Ueff = make_input('U_eff (AC) [V] =', 'z.B. 230 für Netz')
b2_Up   = make_input('Û (AC) [V]    =', 'oder Spitzenwert')
b2_UF   = make_input('U_F Diode [V] =', 'pro Diode, ca. 0.7')
btn_b2  = make_button()
out_b2  = widgets.Output()

def calc_b2(_=None):
    out_b2.clear_output()
    with out_b2:
        try:
            Ueff = p(b2_Ueff.value)
            Up   = p(b2_Up.value)
            UF   = p(b2_UF.value) if b2_UF.value.strip() else 0.7

            if Ueff is None and Up is None:
                print("ℹ️ U_eff oder Û eingeben"); return
            if Up is None:   Up   = Ueff * math.sqrt(2)
            if Ueff is None: Ueff = Up / math.sqrt(2)

            U_avg      = 2 * Up / math.pi
            U_avg_real = U_avg - 2 * UF
            U_sperr    = Up  # Sperrspannung je Diode

            print(f"  Û           = {fmt(Up, 'V')}")
            print(f"  U_eff       = {fmt(Ueff, 'V')}")
            print(f"  Ū (ideal)   = 2Û/π = {fmt(U_avg, 'V')}")
            print(f"  2·U_F       = {fmt(2*UF, 'V')}")
            print(f"  Ū (real)    = {fmt(U_avg_real, 'V')}")
            print(f"  f_Brumm     = 100 Hz  (= 2 · Netzfrequenz)")
            print(f"  U_Sperr     = {fmt(U_sperr, 'V')}  (je Diode mind.)")
            print(f"  Ausnutzung  = 100 %  (beide HW)")
            if U_avg_real < 0:
                print(f"  ⚠️ Ausgangsspannung negativ – Eingangsspannung zu klein!")
        except Exception as e:
            print("❌ Fehler:", e)

btn_b2.on_click(calc_b2)
for fw in [b2_Ueff, b2_Up, b2_UF]:
    fw.observe(calc_b2, names='value')
display(widgets.VBox([b2_Ueff, b2_Up, b2_UF, btn_b2, out_b2]))

---
## 8.4 Siebung mit Kondensator

Der **Siebkondensator** glättet die pulsierende Gleichspannung.
Zwischen den Ladeimpulsen entlädt er sich in die Last.

### Brummspannung (Näherung)
$$U_{Br} \approx \frac{I_{Last}}{f_{Br} \cdot C} = \frac{\bar{U}}{R_{Last} \cdot f_{Br} \cdot C}$$

### Mittlere Ausgangsspannung (mit Siebung)
$$\bar{U}_{DC} \approx \hat{U} - \frac{U_{Br}}{2}$$

### Welligkeit
$$w = \frac{U_{Br}}{\bar{U}_{DC}} \cdot 100\,\%$$

**Faustregeln:**
- Gute Siebung: $w < 1\,\%$
- Ausreichend: $w < 5\,\%$
- $C$ gross wählen → kleine Brummspannung
- Brückengleichrichter: $f_{Br} = 100\,Hz$ → halb so grosser $C$ nötig wie bei M1

<!-- Bild: Siebkondensator Lade- und Entladekurve mit Brummspannung -->
<!-- Dateiname: siebkondensator_brummspannung.png  → Google: "capacitor filter ripple voltage DC power supply waveform labeled" -->

In [ ]:
# ============================================================
# RECHNER 8.4: Siebkondensator und Brummspannung
# ============================================================
display(HTML("<h3>🔢 Rechner 8.4 – Siebkondensator und Brummspannung</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

sib_typ = widgets.ToggleButtons(
    options=['Brücke B2 (f=100Hz)', 'Einweg M1 (f=50Hz)'],
    description='Gleichrichter:', style={'description_width': '140px', 'button_width': '180px'}
)
sib_C    = make_input('C [F]        =', 'z.B. 1000u, 4700u')
sib_I    = make_input('I_Last [A]   =', 'Laststrom')
sib_R    = make_input('R_Last [Ω]   =', 'oder Lastwiderstand')
sib_UBr  = make_input('U_Br [V]     =', 'leer = berechnen')
sib_Up   = make_input('Û [V]        =', 'optional: U_DC berechnen')
btn_sib  = make_button()
out_sib  = widgets.Output()

def calc_sib(_=None):
    out_sib.clear_output()
    with out_sib:
        try:
            C    = p(sib_C.value)
            I    = p(sib_I.value)
            R    = p(sib_R.value)
            UBr  = p(sib_UBr.value)
            Up   = p(sib_Up.value)
            fBr  = 100 if 'B2' in sib_typ.value else 50

            print(f"  f_Brumm = {fBr} Hz")

            # I aus R berechnen wenn nötig
            if I is None and R is not None and Up is not None:
                I = Up / R
                print(f"  I_Last  = Û/R = {fmt(I, 'A')}")
            elif I is None and R is not None and UBr is None:
                print("ℹ️ Û oder I_Last zusätzlich eingeben"); return

            vals = {'C': C, 'U_Br': UBr}
            miss = [k for k, v in vals.items() if v is None]

            if C is not None and I is not None:
                UBr_calc = I / (fBr * C)
                print(f"  U_Br    = I/(f·C) = {fmt(UBr_calc, 'V')}")
                if Up:
                    UDC = Up - UBr_calc / 2
                    w   = UBr_calc / UDC * 100
                    print(f"  Ū_DC    ≈ Û - U_Br/2 = {fmt(UDC, 'V')}")
                    print(f"  Welligkeit w = {w:.3g} %")
                    if   w < 1:  print(f"  → Sehr gute Siebung")
                    elif w < 5:  print(f"  → Ausreichende Siebung")
                    else:        print(f"  ⚠️ Schlechte Siebung – C vergrössern!")

            elif UBr is not None and I is not None:
                C_calc = I / (fBr * UBr)
                print(f"  C = I/(f·U_Br) = {fmt(C_calc, 'F')}")
                print(f"  → Mindestkapazität für U_Br ≤ {fmt(UBr, 'V')}")

            elif C is not None and UBr is not None:
                I_max = UBr * fBr * C
                print(f"  I_max = U_Br·f·C = {fmt(I_max, 'A')}")
                print(f"  → Maximaler Laststrom bei U_Br ≤ {fmt(UBr, 'V')}")

            else:
                print("ℹ️ C + I_Last  oder  U_Br + I_Last  oder  C + U_Br eingeben")

        except Exception as e:
            print("❌ Fehler:", e)

btn_sib.on_click(calc_sib)
sib_typ.observe(calc_sib, names='value')
for fw in [sib_C, sib_I, sib_R, sib_UBr, sib_Up]:
    fw.observe(calc_sib, names='value')
display(widgets.VBox([sib_typ, sib_C, sib_I, sib_R, sib_UBr, sib_Up, btn_sib, out_sib]))

---
## 8.5 Netzteil-Dimensionierung (komplett)

Typische Kette: **Trafo → Gleichrichter → Siebkondensator → Last**

### Vorgehen
1. Gewünschte DC-Spannung $\bar{U}_{DC}$ und Strom $I_{Last}$ festlegen
2. Brummspannung $U_{Br}$ maximal festlegen (z.B. 1% von $\bar{U}_{DC}$)
3. Sekundärspannung Trafo berechnen:
$$U_{2,eff} = \frac{\bar{U}_{DC} + 2 \cdot U_F}{\sqrt{2} \cdot 0.9} \qquad \text{(Brücke B2)}$$
4. Siebkondensator berechnen:
$$C = \frac{I_{Last}}{f_{Br} \cdot U_{Br}}$$
5. Kondensator-Spannungsfestigkeit: mind. $\hat{U} \cdot 1.5$ als Reserve

<!-- Bild: Komplettes Netzteil Blockschaltbild -->
<!-- Dateiname: netzteil_blockschaltbild.png  → Google: "DC power supply block diagram transformer rectifier filter regulator" -->

In [ ]:
# ============================================================
# RECHNER 8.5: Komplette Netzteil-Dimensionierung
# ============================================================
display(HTML("<h3>🔢 Rechner 8.5 – Netzteil-Dimensionierung</h3>"
             "<i>Gewünschte Ausgangsgrössen eingeben</i>"))

nt_UDC  = make_input('Ū_DC [V]      =', 'Gewünschte DC-Spannung')
nt_I    = make_input('I_Last [A]    =', 'Maximaler Laststrom')
nt_w    = make_input('w_max [%]     =', 'Max. Welligkeit, z.B. 1')
nt_UF   = make_input('U_F Diode [V] =', 'ca. 0.7 pro Diode')
btn_nt  = make_button()
out_nt  = widgets.Output()

def calc_nt(_=None):
    out_nt.clear_output()
    with out_nt:
        try:
            UDC = p(nt_UDC.value)
            I   = p(nt_I.value)
            w   = p(nt_w.value)
            UF  = p(nt_UF.value) if nt_UF.value.strip() else 0.7

            if any(x is None for x in [UDC, I]):
                print("ℹ️ Ū_DC und I_Last eingeben"); return

            w_fakt = (w / 100) if w else 0.01  # default 1%
            fBr    = 100  # Brücke B2

            # Brummspannung
            UBr    = w_fakt * UDC

            # Benötigter Spitzenwert
            Up_min = UDC + UBr / 2 + 2 * UF

            # Trafo Sekundärspannung
            U2eff  = Up_min / math.sqrt(2)

            # Siebkondensator
            C_min  = I / (fBr * UBr)

            # Spannungsfestigkeit
            U_fest = Up_min * 1.5

            # Trafo Scheinleistung
            S_trafo = U2eff * I * 1.5  # Faktor 1.5 wegen Stromform

            print(f"--- Ergebnis Netzteil-Dimensionierung (B2) ---")
            print(f"  Ū_DC gewünscht  = {fmt(UDC, 'V')}")
            print(f"  I_Last          = {fmt(I, 'A')}")
            print(f"  Welligkeit max  = {w_fakt*100:.4g} %")
            print(f"\n--- Brummspannung ---")
            print(f"  U_Br max        = {fmt(UBr, 'V')}")
            print(f"\n--- Transformator ---")
            print(f"  Û benötigt      = {fmt(Up_min, 'V')}")
            print(f"  U₂_eff (Trafo)  = {fmt(U2eff, 'V')}")
            print(f"  S_Trafo mind.   = {fmt(S_trafo, 'VA')}")
            print(f"\n--- Siebkondensator ---")
            print(f"  C_min           = {fmt(C_min, 'F')}")
            print(f"  Spg.-festigkeit = {fmt(U_fest, 'V')}  (mind.)")
            print(f"\n--- Dioden ---")
            print(f"  U_F je Diode    = {fmt(UF, 'V')}")
            print(f"  U_Sperr mind.   = {fmt(Up_min, 'V')} je Diode")
            print(f"  I_F mind.       = {fmt(I, 'A')} (Spitzenstrom ~3x höher!)")
        except Exception as e:
            print("❌ Fehler:", e)

btn_nt.on_click(calc_nt)
for fw in [nt_UDC, nt_I, nt_w, nt_UF]:
    fw.observe(calc_nt, names='value')
display(widgets.VBox([nt_UDC, nt_I, nt_w, nt_UF, btn_nt, out_nt]))

---
## 8.6 Vergleich Gleichrichtertypen

| Eigenschaft | Einweg M1 | Brücke B2 |
|-------------|-----------|-----------|
| Anzahl Dioden | 1 | 4 |
| Diodenabfall | $1 \cdot U_F$ | $2 \cdot U_F$ |
| Ū / Û | 0.318 | 0.637 |
| Ū / U_eff | 0.45 | 0.90 |
| Brummfrequenz | 50 Hz | 100 Hz |
| Siebaufwand | Gross | Halb so gross |
| Ausnutzung Trafo | Schlecht | Gut |
| Typische Anwendung | Kleine Signale | Netzteile |

> In der Praxis wird fast immer der **Brückengleichrichter B2** verwendet,
> da die doppelte Brummfrequenz den Siebkondensator halbiert.

---
# Kapitel 9: Thermik

---
## 9.1 Grundlagen – Wärmewiderstand

Wärme fliesst immer von warm nach kalt – analog zum elektrischen Strom!

| Thermisch | Elektrisch | Einheit |
|-----------|-----------|---------|
| Wärmestrom $P_V$ | Strom $I$ | W |
| Temperaturdifferenz $\Delta T$ | Spannung $U$ | K |
| Wärmewiderstand $R_{th}$ | Widerstand $R$ | K/W |

$$\Delta T = P_V \cdot R_{th} \qquad R_{th} = \frac{\Delta T}{P_V}$$

### Thermische Kette (Reihenschaltung)
$$R_{th,ges} = R_{th,jc} + R_{th,cs} + R_{th,sa}$$

| Wärmewiderstand | Bedeutung |
|-----------------|-----------|
| $R_{th,jc}$ | Junction → Case (im Bauteil, aus Datenblatt) |
| $R_{th,cs}$ | Case → Heatsink (Wärmeleitpaste/Pad) |
| $R_{th,sa}$ | Heatsink → Ambient (Kühlkörper) |

<!-- Bild: Thermische Kette Chip-Gehäuse-Kühlkörper-Luft -->
<!-- Dateiname: thermische_kette_rth.png  → Google: "thermal resistance chain junction case heatsink ambient diagram" -->

In [ ]:
# ============================================================
# RECHNER 9.1: Wärmewiderstand und Temperatur
# ============================================================
display(HTML("<h3>🔢 Rechner 9.1 – Wärmewiderstand</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

th_PV   = make_input('P_V [W]      =', 'Verlustleistung')
th_Rth  = make_input('R_th [K/W]   =', 'Wärmewiderstand')
th_dT   = make_input('ΔT [K]       =', 'Temperaturdifferenz')
btn_th  = make_button()
out_th  = widgets.Output()

def calc_th(_=None):
    out_th.clear_output()
    with out_th:
        try:
            PV  = p(th_PV.value)
            Rth = p(th_Rth.value)
            dT  = p(th_dT.value)
            vals = {'P_V': PV, 'R_th': Rth, 'ΔT': dT}
            miss = [k for k, v in vals.items() if v is None]
            if len(miss) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(miss) > 1 else "❌ Ein Feld leer lassen!")
                return
            m = miss[0]
            if   m == 'ΔT':   res = PV * Rth;  print(f"  ΔT   = P_V·R_th = {res:.4g} K")
            elif m == 'R_th': res = dT / PV;   print(f"  R_th = ΔT/P_V   = {res:.4g} K/W")
            else:             res = dT / Rth;  print(f"  P_V  = ΔT/R_th  = {fmt(res, 'W')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_th.on_click(calc_th)
for fw in [th_PV, th_Rth, th_dT]:
    fw.observe(calc_th, names='value')
display(widgets.VBox([th_PV, th_Rth, th_dT, btn_th, out_th]))

---
## 9.2 Kühlkörper-Dimensionierung

### Maximale Sperrschichttemperatur
$$T_j = T_{amb} + P_V \cdot (R_{th,jc} + R_{th,cs} + R_{th,sa})$$

Umgestellt nach benötigtem Kühlkörperwiderstand:
$$R_{th,sa} = \frac{T_{j,max} - T_{amb}}{P_V} - R_{th,jc} - R_{th,cs}$$

**Typische Werte:**
| | $R_{th,cs}$ |
|--|------------|
| Trocken (ohne Paste) | 0.5–1.0 K/W |
| Mit Wärmeleitpaste | 0.1–0.3 K/W |
| Mit Wärmeleitpad | 0.2–0.5 K/W |

In [ ]:
# ============================================================
# RECHNER 9.2: Kühlkörper-Dimensionierung
# ============================================================
display(HTML("<h3>🔢 Rechner 9.2 – Kühlkörper-Dimensionierung</h3>"
             "<i>Bauteilparameter und Betriebsbedingungen eingeben</i>"))

kk_Tj    = make_input('T_j,max [°C]  =', 'Max. Sperrschicht (Datenblatt)')
kk_Tamb  = make_input('T_amb [°C]    =', 'Umgebungstemperatur z.B. 40')
kk_PV    = make_input('P_V [W]       =', 'Verlustleistung')
kk_Rjc   = make_input('R_th,jc [K/W] =', 'Junction→Case (Datenblatt)')
kk_Rcs   = make_input('R_th,cs [K/W] =', 'Case→Sink (Paste ~0.2)')

kk_paste = widgets.Dropdown(
    options=[('-- Wärmeanbindung --', None),
             ('Trocken',             0.8),
             ('Wärmeleitpaste',      0.2),
             ('Wärmeleitpad',        0.4)],
    description='Anbindung:', style={'description_width': '140px'},
    layout=widgets.Layout(width='340px')
)

btn_kk  = make_button()
out_kk  = widgets.Output()

def set_rcs(change):
    if change['new'] is not None:
        kk_Rcs.value = str(change['new'])
kk_paste.observe(set_rcs, names='value')

def calc_kk(_=None):
    out_kk.clear_output()
    with out_kk:
        try:
            Tj   = p(kk_Tj.value);   Tamb = p(kk_Tamb.value)
            PV   = p(kk_PV.value);   Rjc  = p(kk_Rjc.value)
            Rcs  = p(kk_Rcs.value)

            if any(x is None for x in [Tj, Tamb, PV, Rjc]):
                print("ℹ️ T_j,max, T_amb, P_V und R_th,jc eingeben"); return

            Rcs_val = Rcs if Rcs else 0.2
            Rth_ges_max = (Tj - Tamb) / PV
            Rsa_max     = Rth_ges_max - Rjc - Rcs_val
            Tj_ohne_kk  = Tamb + PV * (Rjc + Rcs_val + 50)  # 50 K/W = kein Kühlkörper

            print(f"  T_j,max         = {Tj:.4g} °C")
            print(f"  T_amb           = {Tamb:.4g} °C")
            print(f"  P_V             = {fmt(PV, 'W')}")
            print(f"  R_th,jc         = {Rjc:.4g} K/W")
            print(f"  R_th,cs         = {Rcs_val:.4g} K/W")
            print(f"\n  R_th,ges max    = ΔT/P_V = {Rth_ges_max:.4g} K/W")
            print(f"  R_th,sa max     = {Rsa_max:.4g} K/W  ← Kühlkörper mind. so gut!")

            if Rsa_max < 0:
                print(f"  ⚠️ Kein Kühlkörper ausreichend – Verlustleistung reduzieren!")
            elif Rsa_max < 1:
                print(f"  → Grosser Kühlkörper nötig")
            elif Rsa_max < 5:
                print(f"  → Mittlerer Kühlkörper ausreichend")
            elif Rsa_max < 20:
                print(f"  → Kleiner Kühlkörper oder Gehäuse ausreichend")
            else:
                print(f"  → Kein Kühlkörper nötig (freie Konvektion)")

            # Sicherheitsmarge
            marge = Tj - (Tamb + PV * (Rjc + Rcs_val + Rsa_max))
            print(f"\n  Sicherheitsmarge = {marge:.4g} K  (bei exaktem R_th,sa)")
        except Exception as e:
            print("❌ Fehler:", e)

btn_kk.on_click(calc_kk)
for fw in [kk_Tj, kk_Tamb, kk_PV, kk_Rjc, kk_Rcs]:
    fw.observe(calc_kk, names='value')
display(widgets.VBox([kk_paste, kk_Tj, kk_Tamb, kk_PV, kk_Rjc, kk_Rcs, btn_kk, out_kk]))

---
## 9.3 Widerstandsänderung durch Temperatur (Rückbezug Kapitel 1.5)

Die Verlustleistung im Widerstand erzeugt Wärme und ändert seinen Wert:

$$P_V = I^2 \cdot R \qquad \Delta T = P_V \cdot R_{th}$$

$$R_T = R_{20} \cdot [1 + \alpha_{20} \cdot (T - 20°C)]$$

Dies kann zur **thermischen Instabilität** führen wenn $\alpha > 0$ (Metalle):
Mehr Strom → mehr Wärme → höherer R → weniger Strom (selbstlimitierend ✓)

Bei NTC ($\alpha < 0$):
Mehr Strom → mehr Wärme → kleinerer R → noch mehr Strom → **thermisches Durchgehen** ⚠️

In [ ]:
# ============================================================
# RECHNER 9.3: Thermische Stabilität – Betriebstemperatur
# ============================================================
display(HTML("<h3>🔢 Rechner 9.3 – Betriebstemperatur Widerstand</h3>"
             "<i>R, Strom und thermische Anbindung eingeben</i>"))

ts_R20  = make_input('R₂₀ [Ω]       =', 'Widerstand bei 20°C')
ts_I    = make_input('I [A]         =', 'Betriebsstrom')
ts_Rth  = make_input('R_th [K/W]    =', 'therm. Anbindung')
ts_Tamb = make_input('T_amb [°C]    =', 'Umgebung z.B. 25')
ts_Tmax = make_input('T_max [°C]    =', 'Max. zul. Temp.')
ts_alph = make_input('α₂₀ [1/K]     =', 'z.B. 0.00393 für Cu')
btn_ts  = make_button()
out_ts  = widgets.Output()

def calc_ts(_=None):
    out_ts.clear_output()
    with out_ts:
        try:
            R20  = p(ts_R20.value);  I    = p(ts_I.value)
            Rth  = p(ts_Rth.value);  Tamb = p(ts_Tamb.value)
            Tmax = p(ts_Tmax.value); alph = p(ts_alph.value)

            if any(x is None for x in [R20, I, Rth, Tamb]):
                print("ℹ️ R₂₀, I, R_th und T_amb eingeben"); return

            Tamb_v = Tamb if Tamb else 25.0
            PV     = I**2 * R20
            dT     = PV * Rth
            T_betr = Tamb_v + dT

            print(f"  P_V      = I²·R = {fmt(PV, 'W')}")
            print(f"  ΔT       = P_V·R_th = {dT:.4g} K")
            print(f"  T_betr   = T_amb + ΔT = {T_betr:.4g} °C")

            if alph:
                R_betr = R20 * (1 + alph * (T_betr - 20))
                PV_neu = I**2 * R_betr
                print(f"\n  R bei T_betr = {fmt(R_betr, 'Ω')}")
                print(f"  P_V neu      = {fmt(PV_neu, 'W')}  (Iteration 1)")

            if Tmax:
                reserve = Tmax - T_betr
                print(f"\n  T_max        = {Tmax:.4g} °C")
                print(f"  Reserve      = {reserve:.4g} K")
                if reserve < 0:
                    print(f"  ⚠️ Maximaltemperatur überschritten!")
                elif reserve < 10:
                    print(f"  ⚠️ Wenig Reserve – bessere Kühlung empfohlen")
                else:
                    print(f"  → Thermisch unkritisch")
        except Exception as e:
            print("❌ Fehler:", e)

btn_ts.on_click(calc_ts)
for fw in [ts_R20, ts_I, ts_Rth, ts_Tamb, ts_Tmax, ts_alph]:
    fw.observe(calc_ts, names='value')
display(widgets.VBox([ts_R20, ts_I, ts_Rth, ts_Tamb, ts_Tmax, ts_alph, btn_ts, out_ts]))

---
# Anhang: Interaktive Diagramme

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
%matplotlib inline
print("matplotlib bereit")

---
## Diagramm A: RC-Lade- und Entladekurve

In [ ]:
# ============================================================
# DIAGRAMM A: RC-Lade/Entladekurve – interaktiv
# ============================================================
display(HTML("<h3>📈 Diagramm A – RC Lade- und Entladekurve</h3>"))

dA_R    = make_input('R [Ω]  =', 'z.B. 1k')
dA_C    = make_input('C [F]  =', 'z.B. 100u')
dA_U0   = make_input('U₀ [V] =', 'Quellspannung z.B. 5')
dA_mode = widgets.ToggleButtons(
    options=['Laden', 'Entladen'],
    description='Vorgang:', style={'description_width': '100px', 'button_width': '100px'}
)
btn_dA  = make_button('📈 Zeichnen')
out_dA  = widgets.Output()

def plot_rc(_=None):
    out_dA.clear_output()
    with out_dA:
        try:
            R  = p(dA_R.value); C = p(dA_C.value); U0 = p(dA_U0.value)
            if any(x is None for x in [R, C, U0]):
                print("ℹ️ R, C und U₀ eingeben"); return
            tau = R * C
            t   = np.linspace(0, 5 * tau, 500)

            if dA_mode.value == 'Laden':
                U = U0 * (1 - np.exp(-t / tau))
                I = (U0 / R) * np.exp(-t / tau)
                titel = f'RC-Ladekurve  (τ = {fmt(tau, "s")})'
            else:
                U = U0 * np.exp(-t / tau)
                I = (U0 / R) * np.exp(-t / tau)
                titel = f'RC-Entladekurve  (τ = {fmt(tau, "s")})'

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
            fig.suptitle(titel, fontsize=13)

            # Spannung
            ax1.plot(t * 1000 if tau < 1 else t, U, 'b-', linewidth=2, label='U_C(t)')
            ax1.axhline(U0, color='gray', linestyle='--', alpha=0.5, label=f'U₀ = {U0} V')
            ax1.axhline(0.632 * U0 if dA_mode.value == 'Laden' else 0.368 * U0,
                        color='orange', linestyle=':', alpha=0.7, label='63.2% / 36.8%')
            tau_ms = tau * 1000 if tau < 1 else tau
            ax1.axvline(tau_ms, color='red', linestyle=':', alpha=0.6, label=f'τ = {fmt(tau, "s")}')
            ax1.set_ylabel('Spannung [V]')
            ax1.legend(fontsize=9)
            ax1.grid(True, alpha=0.3)
            ax1.set_ylim(-U0 * 0.05, U0 * 1.15)

            # Strom
            t_plot = t * 1000 if tau < 1 else t
            ax2.plot(t_plot, I * 1000 if max(I) < 0.1 else I,
                     'r-', linewidth=2, label='I(t)')
            ax2.axvline(tau_ms, color='red', linestyle=':', alpha=0.6)
            I_unit = 'mA' if max(I) < 0.1 else 'A'
            ax2.set_ylabel(f'Strom [{I_unit}]')
            t_unit = 'ms' if tau < 1 else 's'
            ax2.set_xlabel(f'Zeit [{t_unit}]')
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)

            # τ Markierungen
            for n in range(1, 6):
                ax1.annotate(f'{n}τ', xy=(n * tau_ms, 0),
                             xytext=(n * tau_ms, -U0 * 0.08),
                             ha='center', fontsize=8, color='red')

            plt.tight_layout()
            plt.show()
            print(f"  τ = {fmt(tau, 's')}  |  5τ = {fmt(5*tau, 's')}  |  I₀ = {fmt(U0/R, 'A')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_dA.on_click(plot_rc)
display(widgets.VBox([dA_mode, dA_R, dA_C, dA_U0, btn_dA, out_dA]))

---
## Diagramm B: Filterkurve (Bode-Diagramm)

In [ ]:
# ============================================================
# DIAGRAMM B: Bode-Diagramm – KORRIGIERT
# ============================================================
display(HTML("<h3>📈 Diagramm B – Filterkurve / Bode-Diagramm</h3>"))

dB_typ  = widgets.ToggleButtons(
    options=['RC-Tiefpass', 'RC-Hochpass', 'Sallen-Key TP 2.Ord'],
    description='Filtertyp:', style={'description_width': '120px', 'button_width': '160px'}
)
dB_R    = make_input('R [Ω]   =', 'z.B. 10k')
dB_C    = make_input('C [F]   =', 'z.B. 10n')
dB_Q    = make_input('Q       =', 'nur Sallen-Key, z.B. 0.707')
btn_dB2 = make_button('📈 Zeichnen')
out_dB2 = widgets.Output()

def plot_bode(_=None):
    out_dB2.clear_output()
    with out_dB2:
        try:
            R = p(dB_R.value); C = p(dB_C.value)
            Q = p(dB_Q.value) if dB_Q.value.strip() else 0.7071
            if any(x is None for x in [R, C]):
                print("ℹ️ R und C eingeben"); return

            fg    = 1 / (2 * math.pi * R * C)
            f     = np.logspace(np.log10(fg / 100), np.log10(fg * 100), 1000)
            ratio = f / fg
            typ   = dB_typ.value

            if typ == 'RC-Tiefpass':
                H     = 1 / np.sqrt(1 + ratio**2)
                phi   = -np.degrees(np.arctan(ratio))
                label = f'RC-Tiefpass  fg = {fmt(fg, "Hz")}'
                phi_fg = -45.0
                phi_lim = (-100, 10)

            elif typ == 'RC-Hochpass':
                H     = ratio / np.sqrt(1 + ratio**2)
                phi   = np.degrees(np.arctan(1 / ratio))
                label = f'RC-Hochpass  fg = {fmt(fg, "Hz")}'
                phi_fg = +45.0
                phi_lim = (-10, 100)

            else:  # Sallen-Key 2. Ordnung
                denom = np.sqrt((1 - ratio**2)**2 + (ratio / Q)**2)
                H     = 1 / denom
                # Korrekte Phase 2. Ordnung: geht von 0° bis −180°
                phi   = -np.degrees(np.arctan2(ratio / Q, 1 - ratio**2))
                label = f'Sallen-Key TP 2.Ord  fg={fmt(fg,"Hz")}  Q={Q:.3g}'
                phi_fg = -90.0   # bei fg ist Phase −90° (nicht −45°!)
                phi_lim = (-200, 10)

            dB_arr = 20 * np.log10(np.maximum(H, 1e-10))

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
            fig.suptitle(f'Bode-Diagramm – {label}', fontsize=12)

            # --- Amplitudengang ---
            ax1.semilogx(f, dB_arr, 'b-', linewidth=2, label='|H(f)|')
            ax1.axvline(fg, color='red', linestyle='--', alpha=0.7,
                        label=f'fg = {fmt(fg, "Hz")}')
            ax1.axhline(-3, color='orange', linestyle=':', alpha=0.7,
                        label='−3 dB')

            # Asymptoten
            if typ == 'RC-Tiefpass':
                ax1.semilogx([fg/100, fg], [0, 0], 'g--', alpha=0.4,
                             linewidth=1, label='Asymptote')
                ax1.semilogx([fg, fg*100], [0, -40], 'g--', alpha=0.4, linewidth=1)
            elif typ == 'RC-Hochpass':
                ax1.semilogx([fg/100, fg], [-40, 0], 'g--', alpha=0.4,
                             linewidth=1, label='Asymptote')
                ax1.semilogx([fg, fg*100], [0, 0], 'g--', alpha=0.4, linewidth=1)
            else:
                # Sallen-Key: −40 dB/Dekade Asymptote
                ax1.semilogx([fg/100, fg], [0, 0], 'g--', alpha=0.4,
                             linewidth=1, label='Asymptote (−40dB/Dek.)')
                ax1.semilogx([fg, fg*100], [0, -80], 'g--', alpha=0.4, linewidth=1)

            ax1.set_ylabel('Verstärkung [dB]')
            ax1.set_ylim(max(np.min(dB_arr) - 5, -90), 10)
            ax1.legend(fontsize=9)
            ax1.grid(True, which='both', alpha=0.3)

            # --- Phasengang ---
            ax2.semilogx(f, phi, 'r-', linewidth=2, label='φ(f)')
            ax2.axvline(fg, color='red', linestyle='--', alpha=0.7)
            ax2.axhline(phi_fg, color='orange', linestyle=':', alpha=0.8,
                        label=f'{phi_fg:.0f}° bei fg')

            ax2.set_ylabel('Phase [°]')
            ax2.set_xlabel('Frequenz [Hz]')
            ax2.set_ylim(phi_lim)
            ax2.yaxis.set_major_locator(ticker.MultipleLocator(45))
            ax2.legend(fontsize=9)
            ax2.grid(True, which='both', alpha=0.3)

            plt.tight_layout()
            plt.show()

            print(f"  fg   = {fmt(fg, 'Hz')}  |  ωg = {fmt(2*math.pi*fg, 'rad/s')}")
            if typ == 'Sallen-Key TP 2.Ord':
                print(f"  Q    = {Q:.4g}")
                print(f"  Phase bei fg: −90°  (2. Ordnung: 0° → −180°)")
                print(f"  Flankensteilheit: −40 dB/Dekade")
            else:
                print(f"  Phase bei fg: {phi_fg:+.0f}°  (1. Ordnung: ±90° gesamt)")
                print(f"  Flankensteilheit: ±20 dB/Dekade")
        except Exception as e:
            print("❌ Fehler:", e)

btn_dB2.on_click(plot_bode)
dB_typ.observe(plot_bode, names='value')
display(widgets.VBox([dB_typ, dB_R, dB_C, dB_Q, btn_dB2, out_dB2]))

---
## Diagramm C: RLC-Schwingkreis Frequenzgang

In [ ]:
# ============================================================
# DIAGRAMM C: RLC Schwingkreis – KORRIGIERT
# ============================================================
display(HTML("<h3>📈 Diagramm C – RLC Schwingkreis Frequenzgang</h3>"))

dC_R   = make_input('R [Ω]  =', 'z.B. 50')
dC_L   = make_input('L [H]  =', 'z.B. 1m')
dC_C   = make_input('C [F]  =', 'z.B. 10u')
dC_typ = widgets.ToggleButtons(
    options=['Reihenschwingkreis', 'Parallelschwingkreis'],
    description='Typ:', style={'description_width': '100px', 'button_width': '180px'}
)
btn_dC = make_button('📈 Zeichnen')
out_dC = widgets.Output()

def plot_rlc(_=None):
    out_dC.clear_output()
    with out_dC:
        try:
            R = p(dC_R.value); L = p(dC_L.value); C = p(dC_C.value)
            if any(x is None for x in [R, L, C]):
                print("ℹ️ R, L und C eingeben"); return

            f0  = 1 / (2 * math.pi * math.sqrt(L * C))
            Q   = (1/R) * math.sqrt(L/C)
            df  = f0 / Q
            fu  = f0 - df/2
            fo  = f0 + df/2

            # Frequenzbereich: weit genug um beide Flanken zu zeigen
            # Mindestens Faktor 50 unter fu und über fo
            f_min = max(fu / 50, f0 / 200)
            f_max = fo * 50
            f     = np.logspace(np.log10(f_min), np.log10(f_max), 2000)
            w     = 2 * math.pi * f
            XL    = w * L
            XC    = 1 / (w * C)

            if dC_typ.value == 'Reihenschwingkreis':
                Z    = np.sqrt(R**2 + (XL - XC)**2)
                Z_dB = 20 * np.log10(Z)
                lbl  = f'Reihenschwingkreis  f₀={fmt(f0,"Hz")}  Q={Q:.3g}'
                # Reihe: Minimum bei f0
                Z_ref = 20 * np.log10(R)
                y_margin = max(np.max(Z_dB) - Z_ref + 5, 20)
                y_lim = (Z_ref - 5, Z_ref + y_margin)
            else:
                # Parallel: Z_parallel korrekt berechnet
                Z_L   = np.sqrt(R**2 + XL**2)       # Impedanz Spulenzweig
                Z_C   = XC                            # Impedanz Kondensator
                # Parallelschaltung beider Zweige
                Z_re  = (R * XC**2 + R * XL**2) / (R**2 + (XL - XC)**2 + 1e-30)
                Z_im  = (XL * XC**2 - XL**2 * XC - R**2 * XC) / (R**2 + (XL - XC)**2 + 1e-30)
                Z     = np.sqrt(Z_re**2 + Z_im**2)
                Z_dB  = 20 * np.log10(np.maximum(Z, 1e-6))
                lbl   = f'Parallelschwingkreis  f₀={fmt(f0,"Hz")}  Q={Q:.3g}'
                # Maximum bei f0 bestimmen – Skalierung darauf ausrichten
                Z_max_dB = np.max(Z_dB)
                Z_min_dB = np.min(Z_dB)
                y_lim = (Z_min_dB - 5, Z_max_dB + 10)

            fig, ax = plt.subplots(figsize=(11, 5))
            ax.semilogx(f, Z_dB, 'b-', linewidth=2, label='Z(f)')

            # f0 Linie
            ax.axvline(f0, color='red', linestyle='--', alpha=0.8,
                       label=f'f₀ = {fmt(f0, "Hz")}')

            # f_u und f_o nur wenn positiv
            if fu > 0:
                ax.axvline(fu, color='orange', linestyle=':', alpha=0.7,
                           label=f'f_u = {fmt(fu, "Hz")}')
            else:
                print(f"  ⚠️ f_u = {fu:.3g} Hz (negativ – Q zu niedrig für symmetrische Bandbreite)")

            ax.axvline(fo, color='orange', linestyle=':', alpha=0.7,
                       label=f'f_o = {fmt(fo, "Hz")}')

            ax.set_title(f'Impedanzkurve – {lbl}', fontsize=12)
            ax.set_xlabel('Frequenz [Hz]')
            ax.set_ylabel('Z [dBΩ]')
            ax.set_ylim(y_lim)
            ax.legend(fontsize=9)
            ax.grid(True, which='both', alpha=0.3)
            plt.tight_layout()
            plt.show()

            print(f"  f₀  = {fmt(f0, 'Hz')}")
            print(f"  Q   = {Q:.4g}")
            print(f"  Δf  = {fmt(df, 'Hz')}")
            if fu > 0:
                print(f"  f_u = {fmt(fu, 'Hz')}")
            else:
                print(f"  f_u = {fu:.3g} Hz  ⚠️ negativ (Q < 0.5 → Bandbreite > 2·f₀)")
            print(f"  f_o = {fmt(fo, 'Hz')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_dC.on_click(plot_rlc)
dC_typ.observe(plot_rlc, names='value')
display(widgets.VBox([dC_typ, dC_R, dC_L, dC_C, btn_dC, out_dC]))

---
## Diagramm D: RL-Einschaltstrom

In [ ]:
# ============================================================
# DIAGRAMM D: RL Einschalt-/Ausschaltkurve
# ============================================================
display(HTML("<h3>📈 Diagramm D – RL Ein-/Ausschaltkurve</h3>"))

dD_R    = make_input('R [Ω]  =', 'z.B. 100')
dD_L    = make_input('L [H]  =', 'z.B. 10m')
dD_U0   = make_input('U₀ [V] =', 'z.B. 12')
dD_mode = widgets.ToggleButtons(
    options=['Einschalten', 'Ausschalten'],
    description='Vorgang:', style={'description_width': '100px', 'button_width': '120px'}
)
btn_dD  = make_button('📈 Zeichnen')
out_dD  = widgets.Output()

def plot_rl(_=None):
    out_dD.clear_output()
    with out_dD:
        try:
            R = p(dD_R.value); L = p(dD_L.value); U0 = p(dD_U0.value)
            if any(x is None for x in [R, L, U0]):
                print("ℹ️ R, L und U₀ eingeben"); return

            tau   = L / R
            I_max = U0 / R
            t     = np.linspace(0, 5 * tau, 500)

            if dD_mode.value == 'Einschalten':
                I  = I_max * (1 - np.exp(-t / tau))
                UL = U0 * np.exp(-t / tau)
                UR = I * R
                titel = f'RL-Einschalten  (τ = {fmt(tau, "s")})'
            else:
                I  = I_max * np.exp(-t / tau)
                UL = -U0 * np.exp(-t / tau)
                UR = I * R
                titel = f'RL-Ausschalten  (τ = {fmt(tau, "s")})'

            t_ms = t * 1000 if tau < 0.1 else t
            t_unit = 'ms' if tau < 0.1 else 's'

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
            fig.suptitle(titel, fontsize=13)

            ax1.plot(t_ms, I * 1000 if I_max < 0.1 else I, 'b-', linewidth=2, label='I(t)')
            ax1.axhline(I_max * 1000 if I_max < 0.1 else I_max,
                        color='gray', linestyle='--', alpha=0.5, label=f'I_max = {fmt(I_max, "A")}')
            ax1.axvline(tau * 1000 if tau < 0.1 else tau,
                        color='red', linestyle=':', alpha=0.7, label=f'τ = {fmt(tau, "s")}')
            I_unit = 'mA' if I_max < 0.1 else 'A'
            ax1.set_ylabel(f'Strom [{I_unit}]')
            ax1.legend(fontsize=9)
            ax1.grid(True, alpha=0.3)

            ax2.plot(t_ms, UL, 'r-', linewidth=2, label='U_L(t)')
            ax2.plot(t_ms, UR, 'g-', linewidth=1.5, alpha=0.7, label='U_R(t)')
            ax2.axhline(0, color='black', linewidth=0.5)
            ax2.axvline(tau * 1000 if tau < 0.1 else tau,
                        color='red', linestyle=':', alpha=0.7)
            ax2.set_ylabel('Spannung [V]')
            ax2.set_xlabel(f'Zeit [{t_unit}]')
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)

            for n in range(1, 6):
                t_mark = n * tau * 1000 if tau < 0.1 else n * tau
                ax1.annotate(f'{n}τ', xy=(t_mark, 0),
                             xytext=(t_mark, -I_max * 0.08 * (1000 if I_max < 0.1 else 1)),
                             ha='center', fontsize=8, color='red')

            plt.tight_layout()
            plt.show()
            print(f"  τ = {fmt(tau, 's')}  |  I_max = {fmt(I_max, 'A')}  |  5τ = {fmt(5*tau, 's')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_dD.on_click(plot_rl)
display(widgets.VBox([dD_mode, dD_R, dD_L, dD_U0, btn_dD, out_dD]))

---
## Diagramm E: Gleichrichter Wellenformen

In [ ]:
# ============================================================
# DIAGRAMM E: Gleichrichter Wellenformen – KORRIGIERT
# ============================================================
display(HTML("<h3>📈 Diagramm E – Gleichrichter Wellenformen</h3>"))

dE_Ueff = make_input('U_eff [V]  =', 'z.B. 12')
dE_UF   = make_input('U_F [V]    =', 'Flussspannung ~0.7')
dE_C    = make_input('C [F]      =', 'Siebkond. z.B. 1000u')
dE_I    = make_input('I_Last [A] =', 'z.B. 0.5')
dE_typ  = widgets.ToggleButtons(
    options=['Einweg M1', 'Brücke B2'],
    description='Typ:', style={'description_width': '100px', 'button_width': '120px'}
)
btn_dE  = make_button('📈 Zeichnen')
out_dE  = widgets.Output()

def plot_gleichrichter(_=None):
    out_dE.clear_output()
    with out_dE:
        try:
            Ueff = p(dE_Ueff.value)
            UF   = p(dE_UF.value) if dE_UF.value.strip() else 0.7
            C    = p(dE_C.value)
            I    = p(dE_I.value)
            if Ueff is None:
                print("ℹ️ U_eff eingeben"); return

            Up   = Ueff * math.sqrt(2)
            f    = 50
            t    = np.linspace(0, 2/f, 2000)
            U_ac = Up * np.sin(2 * math.pi * f * t)

            if dE_typ.value == 'Einweg M1':
                U_rect = np.where(U_ac > UF, U_ac - UF, 0.0)
                fBr    = f
                UF_tot = UF
                titel  = f'Einweggleichrichter M1  (Û = {fmt(Up, "V")})'
            else:
                U_rect = np.abs(U_ac) - 2 * UF
                U_rect = np.where(U_rect > 0, U_rect, 0.0)
                fBr    = 2 * f
                UF_tot = 2 * UF
                titel  = f'Brückengleichrichter B2  (Û = {fmt(Up, "V")})'

            U_avg = np.mean(U_rect)

            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(t * 1000, U_ac, 'b-', alpha=0.35, linewidth=1.5,
                    label='U_AC (Eingang)')
            ax.plot(t * 1000, U_rect, 'r-', linewidth=2,
                    label='U_rect (nach Gleichrichter)')

            warnings = []

            if C and I:
                UBr  = I / (fBr * C)
                UDC  = Up - UF_tot - UBr / 2

                if UBr >= Up - UF_tot:
                    # Welligkeit grösser als gesamte Spannung – C viel zu klein
                    warnings.append(
                        f"⚠️ C zu klein / I zu gross: Welligkeit ({fmt(UBr,'V')}) ≥ "
                        f"Spitzenwert ({fmt(Up-UF_tot,'V')}) → Siebung unwirksam!"
                    )
                    # DC-Linie trotzdem zeigen, aber auf 0 begrenzen
                    UDC_plot  = max(UDC, 0)
                    UBr_plot  = min(UBr, Up - UF_tot)
                    U_low     = max(UDC - UBr / 2, 0)
                else:
                    UDC_plot = UDC
                    UBr_plot = UBr
                    U_low    = UDC - UBr / 2

                    if U_low < 0:
                        warnings.append(
                            f"⚠️ Untere Welligkeit ({fmt(U_low,'V')}) unter 0 V – "
                            f"C vergrössern oder I reduzieren!"
                        )
                        U_low = 0  # Auf 0 begrenzen für Darstellung

                ax.axhline(UDC_plot, color='green', linestyle='--', linewidth=1.8,
                           label=f'Ū_DC ≈ {UDC_plot:.3g} V')
                ax.axhline(U_low, color='purple', linestyle=':',
                           linewidth=1.2, label=f'U_min ≈ {U_low:.3g} V')
                ax.fill_between(t * 1000,
                                U_low,
                                min(UDC_plot + UBr_plot / 2, Up - UF_tot),
                                alpha=0.15, color='green', label='Welligkeit')

                w_pct = (UBr / UDC * 100) if UDC > 0 else float('inf')
                print(f"  Û           = {fmt(Up, 'V')}")
                print(f"  Ū_DC        = {fmt(UDC, 'V')}")
                print(f"  U_Br        = {fmt(UBr, 'V')}")
                if math.isfinite(w_pct):
                    print(f"  Welligkeit  = {w_pct:.3g} %")
                else:
                    print(f"  Welligkeit  = nicht definiert (U_DC ≤ 0)")
            else:
                U_avg_th = Up / math.pi - UF_tot/2 if dE_typ.value == 'Einweg M1' \
                           else 2*Up/math.pi - UF_tot
                ax.axhline(U_avg_th, color='orange', linestyle='--', alpha=0.8,
                           label=f'Ū (ohne Siebung) = {U_avg_th:.3g} V')
                print(f"  Û      = {fmt(Up, 'V')}")
                print(f"  Ū      = {fmt(U_avg_th, 'V')}")

            ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
            ax.set_title(titel, fontsize=12)
            ax.set_xlabel('Zeit [ms]')
            ax.set_ylabel('Spannung [V]')
            ax.legend(fontsize=9, loc='upper right')
            ax.grid(True, alpha=0.3)

            # Y-Achse immer vernünftig skalieren – nie unter −Up*0.15
            ax.set_ylim(-Up * 0.15, Up * 1.2)

            plt.tight_layout()
            plt.show()

            for w in warnings:
                print(w)

        except Exception as e:
            print("❌ Fehler:", e)

btn_dE.on_click(plot_gleichrichter)
dE_typ.observe(plot_gleichrichter, names='value')
display(widgets.VBox([dE_typ, dE_Ueff, dE_UF, dE_C, dE_I, btn_dE, out_dE]))